# Step 04 — semantic enrichment

Step 03 answered *where the buildings are and how big they are*. This notebook
answers **what each one is for**.

| | |
|---|---|
| **Reads** | `data/output/03_alkis_by_building.gpkg` (426 MB, 869,316 buildings), `data/output/01_all_buildings_osm.gpkg` (161 MB, 517,196 OSM footprints) and `data/output/01_all_pois.gpkg` (26,999 POIs, for the presence test and the join) |
| | `data/input/FS_LN_03_NI_260101.gpkg` (ALKIS Landnutzung, statewide, clipped here) and `data/output/01_landuse_osm.gpkg` |
| **Writes** | `data/experimental_extract/04_buildings_labelled.gpkg` + `.qml` — every building with the drop decision, for QGIS |
| | `data/output/04_buildings_enriched.gpkg` — layers `buildings`, `building_pois`, `pois_unassigned`: the result of this step |
| | `data/experimental_extract/04_buildings_kept.gpkg` + `.qml` — only what survives, with its POIs, for QGIS |
| | `data/experimental_extract/04_poi_assignments.gpkg` — one line per building–POI pair, for QGIS |
| **Needs** | `geopandas`, `pyogrio` |

## State of this notebook

Built **one step at a time**, each run and inspected before the next is written.

| step | | status |
|---|---|---|
| **04.1** | **Slim the layer** — 30 columns → 18, addresses combined | **implemented** |
| **04.2** | **Function labels + activity map** | **implemented** |
| **04.3** | **Fill the gaps with OSM footprints** — the buildings ALKIS does not have | **implemented** |
| **04.4** | **Drop what is not a place of activity** — list 1 (29 ALKIS classes) outright, then place every POI on an actual building, then list 2 (19 classes, residential included) unless a POI or site is on them; the OSM gap fill is judged the same way by its `building=*` tag; class 2000 below 100 m² goes unless its OSM twin or a POI says otherwise | **implemented** |
| **04.5** | **The POI join** — which POI is on which building, with the shares; sites onto every kept building inside; units share their mall | **implemented** |
| **04.6** | **Write the enriched layer** | **implemented** |

## Which input layer, and why

`03_alkis_by_building.gpkg` — **one row per ALKIS object**, not per LoD2 part.
Step 03 writes both and the part layer is the authoritative one, but for
semantics the building is the right unit: `function`, `name` and the address are
attributes of the ALKIS object, so every part of a building carries identical
values. Measured in step 03: **0 of 869,316 buildings have parts that disagree
on `function`.** Nothing semantic is lost by working at building level, and a
POI sitting inside a hospital belongs to the hospital rather than to whichever
wing happens to contain it.

The part layer is there to fall back on if a building-level answer ever looks
wrong.

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Same reason as steps 01-03: a notebook's working directory is not necessarily
# its own folder, so Path('..') is unreliable.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import time
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio

from config import (
    ALKIS_BY_BUILDING_FILE, ALKIS_SLIM_COLS, ALKIS_ADDRESS_PARTS, TARGET_CRS,
    BUILDING_FUNCTION_CODELIST_FILE, ALKIS_ACTIVITY_MAP_FILE, ALKIS_ACTIVITY_SEP,
    LABELLED_INSPECT_FILE, KEPT_INSPECT_FILE, EXPERIMENTAL_DIR,
    ALL_BUILDINGS_OSM_FILE, LOD2_REGION_TILES_FILE,
    OSM_GAP_MAX_ALKIS_COVERAGE, OSM_GAP_EXCLUDE_BUILDING_TAGS,
    OSM_GAP_FUNCTION_CODE, OSM_GAP_LABEL_EN, OSM_GAP_AGS_MAX_DISTANCE_M,
    OSM_GAP_EXPECTED_SHARE_PCT, OSM_GAP_FLOOR_HEIGHT_M, OSM_GAP_DEFAULT_FLOORS,
    ALKIS_HOME_ONLY_ACTIVITIES,
    ALL_POIS_FILE, ALKIS_DROP_ALWAYS, ALKIS_DROP_UNLESS_POI, POI_SITE_RESCUE_MIN_AREA_M2,
    OSM_DROP_ALWAYS, OSM_DROP_UNLESS_POI,
    SIZE_FLOOR_M2, ALKIS_NAME_NOT_EVIDENCE, ALKIS_SIZE_FLOOR_EVIDENCE_M2, OSM_TWIN_STRUCTURE_TAGS, OSM_TWIN_ACTIVITY_TAGS,
    ALKIS_LANDUSE_FILE, ALKIS_LANDUSE_LABELS_EN, ALKIS_LANDUSE_DETAIL_EN, ALKIS_LANDUSE_DROP, ALKIS_LANDUSE_RULE_CLASSES, LANDUSE_OSM_FILE,
    POI_SNAP_MAX_DISTANCE_M, POI_BUILDING_BOUND_MIN_INSIDE_SHARE,
    POI_SPLIT_AREA_FALLBACK_M2, POI_CONTAINER_USES,
    ENRICHED_BUILDINGS_FILE, POI_ASSIGNMENTS_FILE, OUTPUT_DIR,
)
from lib.checks import require_file, require_non_empty, require_crs, require_unique, require_cols

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

print('Root :', ROOT_DIR)
print('Input:', ALKIS_BY_BUILDING_FILE.name, '+', ALL_BUILDINGS_OSM_FILE.name, '+', ALL_POIS_FILE.name)

Root : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
Input: 03_alkis_by_building.gpkg + 01_all_buildings_osm.gpkg + 01_all_pois.gpkg


## 1. Input contract

The building layer must be what step 03 promised: one row per `alkis_id`, flat
2D geometry in the target CRS, with `function` populated everywhere.

`function` being 100 % filled is the one that matters — it is the parameter the
whole classification turns on, and a gap there would propagate silently into
every activity assignment.

In [2]:
require_file(ALKIS_BY_BUILDING_FILE, 'ALKIS buildings (step 03)')

print('Reading (~20 s) ...', flush=True)
t0 = time.perf_counter()
bld = gpd.read_file(ALKIS_BY_BUILDING_FILE, layer='buildings')
print(f'  ok  {len(bld):,} rows x {len(bld.columns)} columns  '
      f'[{time.perf_counter() - t0:,.0f}s]')

require_non_empty(bld, 'buildings')
require_crs(bld, TARGET_CRS, 'buildings')
require_unique(bld, 'alkis_id', 'buildings')

if bld.geometry.has_z.any():
    raise AssertionError('geometry still carries Z - step 03 should have flattened it')
print(f'  ok  geometry is 2D: {bld.geometry.geom_type.value_counts().to_dict()}')

_nf = int(bld['function'].isna().sum())
if _nf:
    raise AssertionError(
        f'{_nf:,} buildings have no `function` code. Everything in this notebook '
        'keys off it, so a gap here would propagate silently.'
    )
print(f'  ok  function: 100 % filled, {bld["function"].nunique()} distinct codes')

  ok  03_alkis_by_building.gpkg (424.7 MB)
Reading (~20 s) ...


  ok  869,316 rows x 32 columns  [6s]
  ok  buildings: 869,316 rows
  ok  buildings: CRS EPSG:25832
  ok  buildings.alkis_id: unique and non-null (869,316)


  ok  geometry is 2D: {'MultiPolygon': 869316}
  ok  function: 100 % filled, 88 distinct codes


## 2. What arrived

Two things worth looking at before dropping anything.

**The function codes.** These are AdV `Gebaeudefunktion` values in the form
`<AAA object class>_<function>`. The prefix matters: `31001` is `AX_Gebaeude`,
an actual building, and the `51xxx` classes are other structures — canopies,
installations. Step 03 measured **11.57 % of rows are not `31001`**, which is
the single most important thing to know before joining the reference tables in
04.2: if they only cover `31001_*`, roughly 100,000 rows fall out of the join
with no error raised.

**Fill rates.** `name` and the address columns are sparse, and how sparse
decides what is possible later — an OSM name match can only reach the buildings
that have a name.

In [3]:
print('AAA object class prefixes:')
_pref = bld['function'].str.slice(0, 5).value_counts()
for p, c in _pref.items():
    print(f'  {p}   {c:>9,}   ({100 * c / len(bld):>5.2f} %)')
print(f'  -> not 31001: {len(bld) - _pref.get("31001", 0):,} '
      f'({100 * (1 - _pref.get("31001", 0) / len(bld)):.2f} %)')

print()
print('the 12 most common function codes:')
for f, c in bld['function'].value_counts().head(12).items():
    print(f'  {f:<14} {c:>9,}  ({100 * c / len(bld):>5.2f} %)')

print()
print('fill rates of the columns that are about to be kept or dropped:')
for c in sorted(bld.columns):
    if c == 'geometry':
        continue
    nn = bld[c].notna().sum()
    keep = 'keep' if c in ALKIS_SLIM_COLS else ('-> address' if c in ALKIS_ADDRESS_PARTS else 'DROP')
    print(f'  {c:<20} {100 * nn / len(bld):>6.2f} %  {str(bld[c].dtype):<8}  {keep}')

AAA object class prefixes:


  31001     768,756   (88.43 %)
  51009      94,582   (10.88 %)
  51002       3,669   ( 0.42 %)
  51003       1,988   ( 0.23 %)
  51001         254   ( 0.03 %)
  51006          50   ( 0.01 %)
  51007          17   ( 0.00 %)
  -> not 31001: 100,560 (11.57 %)

the 12 most common function codes:


  31001_2000       369,675  (42.52 %)
  31001_1000       327,264  (37.65 %)
  51009_1610        94,571  (10.88 %)
  31001_2720        18,850  ( 2.17 %)
  31001_2100        11,849  ( 1.36 %)
  31001_2010         8,631  ( 0.99 %)
  31001_2500         7,806  ( 0.90 %)
  31001_1120         5,793  ( 0.67 %)
  31001_1210         4,197  ( 0.48 %)
  31001_3000         3,301  ( 0.38 %)
  51003_1201         1,784  ( 0.21 %)
  51002_1250         1,772  ( 0.20 %)

fill rates of the columns that are about to be kept or dropped:
  ags                  100.00 %  str       keep
  alkis_id             100.00 %  str       keep
  area_m2              100.00 %  float64   keep
  city                 100.00 %  str       keep
  created_on           100.00 %  str       DROP
  elev_ground_min_m    100.00 %  float64   DROP
  elev_top_max_m       100.00 %  float64   DROP
  function             100.00 %  str       keep
  functions_all          0.00 %  object    DROP
  height_eaves_max_m   100.00 %  float64   keep

  house_number          34.91 %  str       -> address
  is_multipart         100.00 %  bool      keep
  n_functions          100.00 %  int64     DROP
  n_parts              100.00 %  int64     keep
  n_roof_clipped       100.00 %  int32     DROP
  n_roof_faces         100.00 %  int32     DROP
  n_surfaces           100.00 %  int32     DROP
  n_uuid               100.00 %  int64     DROP
  name                   0.58 %  str       keep


  plan_acquired_on     100.00 %  str       DROP
  roof_area_m2         100.00 %  float64   DROP
  roof_code            100.00 %  int64     DROP
  roof_fallback_any    100.00 %  bool      keep
  roof_manual_any      100.00 %  bool      keep
  roof_shape           100.00 %  str       keep
  street                34.91 %  str       -> address
  volume_3d_m3         100.00 %  float64   keep
  volume_old_m3        100.00 %  float64   keep
  volume_ratio         100.00 %  float64   keep


## 3. Combine the address, then slim to 18 columns

`street` and `house_number` become one `address` field — `"Aachener Straße 12"`
— falling back to the street alone when the number is missing, and to NULL when
neither exists.

**Expect it empty on about half the layer.** `street` and `house_number` are
~50 % filled, so anything downstream that wants address matching can only ever
reach half the building stock. That is a property of ALKIS, not of this step,
and it is better known now than discovered during a join.

What goes, and why, is recorded in `ALKIS_SLIM_COLS` in `config.py`. The two
worth restating here:

* **`n_functions` / `functions_all`** — always `1` and always NULL, because
  `function` belongs to the ALKIS object. They were guards for a disagreement
  that cannot occur in this region.
* **`elev_ground_min_m` / `elev_top_max_m`** — sea-level elevations. Capacity
  depends on heights above ground, which are kept. Note this drops the only
  topography signal in the layer; restore `elev_ground_min_m` if a later step
  wants terrain.

In [4]:
_street, _house = ALKIS_ADDRESS_PARTS
_s = bld[_street].fillna('').astype(str).str.strip()
_h = bld[_house].fillna('').astype(str).str.strip()
addr = (_s + ' ' + _h).str.strip()
bld['address'] = addr.where(addr != '', None)

print(f'  ..  {_street:<13} {100 * bld[_street].notna().mean():>6.2f} % filled')
print(f'  ..  {_house:<13} {100 * bld[_house].notna().mean():>6.2f} % filled')
print(f'  ..  address       {100 * bld["address"].notna().mean():>6.2f} % filled')
_street_only = int((bld[_street].notna() & bld[_house].isna()).sum())
print(f'  ..  street but no number: {_street_only:,}')
print()
print('  examples:')
for v in bld.loc[bld['address'].notna(), 'address'].head(5):
    print(f'      {v}')

missing = [c for c in ALKIS_SLIM_COLS if c not in bld.columns]
if missing:
    raise AssertionError(f'ALKIS_SLIM_COLS names columns that do not exist: {missing}')
dropped = [c for c in bld.columns if c not in ALKIS_SLIM_COLS]

slim = bld[ALKIS_SLIM_COLS].copy()
print()
print(f'  ..  {len(bld.columns)} columns -> {len(slim.columns)} '
      f'({len(dropped)} dropped)')
print(f'  ..  dropped: {", ".join(sorted(dropped))}')
require_unique(slim, 'alkis_id', 'slim buildings')

print()
print(slim.drop(columns='geometry').head(5).to_string(index=False))
print()
print('  the layer that goes into 04.2:')
for c in slim.columns:
    if c == 'geometry':
        continue
    nn = slim[c].notna().sum()
    print(f'      {c:<20} {100 * nn / len(slim):>6.2f} % filled   {slim[c].dtype}')

  ..  street         34.91 % filled
  ..  house_number   34.91 % filled
  ..  address        34.91 % filled
  ..  street but no number: 0

  examples:
      Ziegelofen 7
      Ackerweg 2
      Stieglitzweg 3
      Schulring 21
      Schaftrift 13



  ..  33 columns -> 19 (14 dropped)
  ..  dropped: created_on, elev_ground_min_m, elev_top_max_m, functions_all, house_number, n_functions, n_roof_clipped, n_roof_faces, n_surfaces, n_uuid, plan_acquired_on, roof_area_m2, roof_code, street
  ok  slim buildings.alkis_id: unique and non-null (869,316)



        alkis_id   function name        address                city      ags  area_m2  volume_3d_m3  volume_old_m3  volume_ratio  height_top_max_m  height_top_avg_m  height_eaves_max_m   roof_shape  roof_manual_any  roof_fallback_any  n_parts  is_multipart
DENIAL01000000Fg 51002_1250  NaN            NaN Braunschweig, Stadt 03101000     4.00          14.0           14.0        1.0000             3.500             3.500               3.500 PolyFlatRoof            False               True        1         False
DENIAL01000000Fh 51002_1250  NaN            NaN Braunschweig, Stadt 03101000     4.00          14.0           14.0        1.0000             3.500             3.500               3.500 PolyFlatRoof            False               True        1         False
DENIAL01000002A0 31001_1000  NaN   Ziegelofen 7 Braunschweig, Stadt 03101000   115.68         673.3          840.7        1.2486             7.404             7.268               5.537    GableRoof            False              F

## 4. Attach the labels and the activities

Two reference tables, both keyed on the same `31001_1000` form as the layer, so
both are plain left joins:

| file | rows | gives |
|---|---|---|
| `building_function_codelist_de_en.csv` | 301 | `label_en` — what the code *means* (the German `label_de` column is left in the file but not joined) |
| `alkis_building_activity_map.csv` | 280 | `activities` — what *happens* there |

The activity map is the consequential one: `activities` is what the
redistribution eventually weights, so a code missing from it yields a building
with no activity, which would drop out silently rather than raise anything.

**Measured coverage: 88 of 88 codes present in both.** Nothing falls out. The
check below asks the question in that direction anyway — *which of OUR codes are
missing?* — rather than the reassuring but useless "how much of the reference do
we use?".

The activity map was converted once from `alkis_building_activity_map.xlsx`
(kept beside it) so the pipeline needs no `openpyxl` and the table stays
greppable. Both CSVs carry a BOM, hence `utf-8-sig`: read as plain `utf-8` the
first column comes back with a `\ufeff` prefix and the join matches nothing.

The 14 atomic activities in use are `business, daycare, early_education,
education, errands, home, leisure, lessons, meetup, other, shopping, sports,
unspecified, work`. Note `unspecified` and `other` are real values in the
source, not placeholders — a building can legitimately end up with nothing
informative.

In [5]:
require_file(BUILDING_FUNCTION_CODELIST_FILE, 'AdV function codelist')
require_file(ALKIS_ACTIVITY_MAP_FILE, 'ALKIS activity map')

codes = pd.read_csv(BUILDING_FUNCTION_CODELIST_FILE, encoding='utf-8-sig', dtype=str)
acts = pd.read_csv(ALKIS_ACTIVITY_MAP_FILE, encoding='utf-8-sig', dtype=str)

# Both are keyed tables, so a duplicate key would quietly multiply rows in the
# join. Asserted, not assumed.
require_unique(codes, 'function', 'codelist')
if 'gfk_code' not in acts.columns:
    raise AssertionError(f'activity map columns are {list(acts.columns)}, '
                         "expected a 'gfk_code' column - check the BOM")
acts = acts.rename(columns={'gfk_code': 'function'})
require_unique(acts, 'function', 'activity map')
print(f'  ..  codelist {len(codes)} codes | activity map {len(acts)} codes')

# --- coverage, asked in the direction that can hurt -------------------------
ours = (slim.groupby('function')
        .agg(n=('alkis_id', 'size'), vol=('volume_3d_m3', 'sum')))
for tbl, name in ((codes, 'codelist'), (acts, 'activity map')):
    missing = ours[~ours.index.isin(set(tbl['function']))]
    if len(missing):
        print(f'  !!  {len(missing)} of our {len(ours)} codes are ABSENT from the '
              f'{name}: {missing["n"].sum():,} buildings '
              f'({100 * missing["n"].sum() / len(slim):.2f} %), '
              f'{missing["vol"].sum() / 1e6:,.1f}M m3 '
              f'({100 * missing["vol"].sum() / slim["volume_3d_m3"].sum():.2f} % of volume)')
        print(missing.sort_values('n', ascending=False).head(10).to_string())
    else:
        print(f'  ok  all {len(ours)} of our codes are present in the {name}')

# --- join --------------------------------------------------------------------
before = len(slim)
slim = slim.merge(codes[['function', 'label_en']], on='function', how='left')
slim = slim.merge(acts[['function', 'activities']], on='function', how='left')
if len(slim) != before:
    raise AssertionError(f'the joins changed the row count {before:,} -> {len(slim):,} '
                         '- a reference table has duplicate keys')
print(f'  ok  joined, still {len(slim):,} rows')

for c in ('label_en', 'activities'):
    nn = slim[c].notna().sum()
    print(f'  ..  {c:<12} {100 * nn / len(slim):>6.2f} % filled')

# The activity list, exploded, so it can be counted per atomic activity.
slim['n_activities'] = (slim['activities'].fillna('')
                        .str.split(ALKIS_ACTIVITY_SEP)
                        .map(lambda xs: len([x for x in xs if x.strip()])))
print()
print('  ..  buildings per atomic activity:')
_ex = (slim[['alkis_id', 'volume_3d_m3', 'activities']]
       .assign(a=slim['activities'].fillna('').str.split(ALKIS_ACTIVITY_SEP))
       .explode('a'))
_ex['a'] = _ex['a'].str.strip()
_ex = _ex[_ex['a'] != '']
_t = _ex.groupby('a').agg(buildings=('alkis_id', 'size'),
                          volume_Mm3=('volume_3d_m3', lambda v: v.sum() / 1e6))
_t['pct_vol'] = (100 * _t['volume_Mm3'] / (slim['volume_3d_m3'].sum() / 1e6)).round(2)
print(_t.sort_values('buildings', ascending=False).round(1).to_string())
_none = int(slim['n_activities'].eq(0).sum())
print(f'\n  ..  buildings with NO activity at all: {_none:,} '
      f'({100 * _none / len(slim):.3f} %)')

  ok  building_function_codelist_de_en.csv (0.0 MB)
  ok  alkis_building_activity_map.csv (0.0 MB)
  ok  codelist.function: unique and non-null (301)
  ok  activity map.function: unique and non-null (280)
  ..  codelist 301 codes | activity map 280 codes


  ok  all 88 of our codes are present in the codelist
  ok  all 88 of our codes are present in the activity map


  ok  joined, still 869,316 rows
  ..  label_en     100.00 % filled
  ..  activities   100.00 % filled



  ..  buildings per atomic activity:


                 buildings  volume_Mm3  pct_vol
a                                              
work                536702       366.7     54.1
business            402230       288.4     42.6
home                339246       324.2     47.9
meetup              338084       325.2     48.0
leisure             102430        32.1      4.7
errands              18986        62.9      9.3
shopping             15441        51.5      7.6
education             2710        17.3      2.6
lessons               1597         2.1      0.3
sports                 765         5.6      0.8
early_education        186         1.0      0.2
other                  105         0.1      0.0

  ..  buildings with NO activity at all: 0 (0.000 %)


## 5. Per-code summary — which building types are worth keeping?

Printed here for the discussion, not written to disk. This is the decision material, not a decision. One row per `function` code with
everything needed to judge it: what it is, what activities it has been assigned,
how many buildings, how much of the region's volume, and a physical profile
(median area, median height, share of flat roofs).

**The case that needs your eye most:**

```
51009_1610   Überdachung / canopy   →   activities: work;leisure
             94,571 buildings, 10.9 % of the layer
```

A canopy is a roof over a petrol forecourt or a bus stop. Nothing happens
*inside* it, because it has no inside — yet it carries `work;leisure`, so in a
volume-proportional redistribution those 94,571 structures compete with real
buildings for worker and leisure demand.

It is not obviously wrong to keep them: a covered loading bay at a depot is
arguably part of a workplace. But it is a decision, and the numbers below plus
the QGIS layer are what it should be made from rather than by inheriting whatever
the old pipeline did.

The `median_height` and `pct_flat_roof` columns are there because they separate
"structure" from "building" physically: a canopy is low and flat, a workshop is
not.

In [6]:
prof = slim.groupby(['function', 'label_en', 'activities'],
                    dropna=False).agg(
    n_buildings=('alkis_id', 'size'),
    volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 3)),
    median_area_m2=('area_m2', 'median'),
    median_height_m=('height_top_max_m', 'median'),
    median_volume_m3=('volume_3d_m3', 'median'),
    pct_flat_roof=('roof_shape', lambda s: round(100 * (s == 'PolyFlatRoof').mean(), 1)),
    pct_named=('name', lambda s: round(100 * s.notna().mean(), 2)),
).reset_index()
prof['pct_buildings'] = (100 * prof['n_buildings'] / len(slim)).round(3)
prof['pct_volume'] = (100 * prof['volume_Mm3'] /
                      (slim['volume_3d_m3'].sum() / 1e6)).round(3)
prof = prof.sort_values('n_buildings', ascending=False)

cols = ['function', 'label_en', 'activities', 'n_buildings', 'pct_buildings',
        'volume_Mm3', 'pct_volume', 'median_area_m2', 'median_height_m',
        'pct_flat_roof', 'pct_named']
print('the 20 codes with the most buildings:')
print(prof[cols].head(20).to_string(index=False))

print(f'\n  ..  {len(prof)} codes in total (printed here only - nothing is written)')

print()
print('  ..  the non-31001 classes, which are structures rather than buildings:')
_n31 = prof[~prof['function'].str.startswith('31001')]
print(_n31[cols].head(12).to_string(index=False))
print(f'\n      {len(_n31)} codes, {_n31["n_buildings"].sum():,} buildings '
      f'({100 * _n31["n_buildings"].sum() / len(slim):.2f} %), '
      f'{_n31["volume_Mm3"].sum():,.1f}M m3 '
      f'({100 * _n31["volume_Mm3"].sum() / (slim["volume_3d_m3"].sum() / 1e6):.2f} % of volume)')

the 20 codes with the most buildings:
  function                                                        label_en                                 activities  n_buildings  pct_buildings  volume_Mm3  pct_volume  median_area_m2  median_height_m  pct_flat_roof  pct_named
31001_2000                              Buildings for business or commerce                              work;business       369675         42.525     103.126      15.227          28.670           2.8730           74.3       0.05
31001_1000                                           residential buildings                                home;meetup       327264         37.646     299.945      44.287         109.030           8.5660           15.7       0.09
51009_1610                                                          canopy                               work;leisure        94571         10.879      10.098       1.491          10.100           3.8720           77.6       0.00
31001_2720                     Agricultural an

## 6. Fill the gaps in ALKIS with OSM footprints

ALKIS/LoD2 is the authoritative building stock, but it is not complete. OSM has
footprints where ALKIS has no record — buildings finished since the LoD2
release, sheds and huts the cadastre never took up, and a few hundred buildings
outside the LoD2 tile set altogether. Left alone, those are holes in the map and
in the redistribution. Here they are filled from `01_all_buildings_osm.gpkg`.

### How "no ALKIS record" is decided

By **footprint coverage**, not by a touch test: the share of each OSM
footprint's area that ALKIS buildings cover. A touch test would call an OSM shed
"present" because the neighbour's wall clips its corner by a few centimetres.

Measured, the coverage is sharply bimodal — most OSM footprints are either
untouched by ALKIS or more than half covered — and the threshold in
`OSM_GAP_MAX_ALKIS_COVERAGE` sits on the valley floor between the two modes.
The histogram is printed below so a re-run on other data shows whether the
valley is still where the threshold is. **The 10–50 % band is not filled**: an
OSM polygon a third covered by ALKIS is almost always the same building drawn
differently, and filling it would put a second footprint on top of the first.

### What the filled rows look like

They join the layer with the **same columns as the ALKIS rows**, plus:

| column | ALKIS rows | OSM rows |
|---|---|---|
| `source` | `alkis` | `osm` |
| `building_id` | the `alkis_id` | the OSM `bld_id` (`way/4708003`) — the one key that is unique across the whole layer |
| `function` | AdV code | **`OSM`** — a single synthetic class, so QGIS shows the whole fill as one legend entry |
| `label_en` | codelist | *Building mapped in OSM, no ALKIS record* |
| `osm_building` | NULL | the OSM `building=*` tag, for splitting the fill further |
| `osm_levels`, `osm_height_m` | NULL | `building:levels` / `height` where OSM has them (rare) |
| `ags`, `city` | ALKIS | from the **nearest ALKIS building**, so the administrative key stays in ALKIS's vocabulary |
| `area_m2` | ALKIS | the footprint area |
| `volume_3d_m3`, `volume_old_m3` | LoD2 | **estimated**, the previous pipeline's way: footprint × height, where height is the OSM `height` tag, else `building:levels` × `OSM_GAP_FLOOR_HEIGHT_M`, else `OSM_GAP_DEFAULT_FLOORS` × that. Both columns get the same value, `volume_ratio` is 1 |
| `height_top_max_m`, `height_top_avg_m` | LoD2 | the same estimated height |
| `roof_*`, `height_eaves_max_m` | LoD2 | **NULL** — there is no LoD2 model to measure |
| `activities` | activity map | **NULL** — no AdV code to map from |

Two consequences, both deliberate and both recorded in `config.py`:

* **The volume is an estimate, and a low one.** OSM has `building:levels` on
  about 5 % of these rows and `height` on 1 %; the rest get one floor of
  2 m, against a median of 7.3 m on the ALKIS buildings that survive. The rows
  are in the model with a small weight rather than none; `source == 'osm'`
  marks them so the weight can be revisited.
* **No activities, so the drop judges them by their OSM `building=*` tag
  instead** (section 7): `roof`, `shed`, `garage` go like the ALKIS structures,
  `house`, `apartments` and the bare `yes` go unless a POI or site is on them. An OSM `building=*` → activity map for the kept ones
  is still to be written.


In [7]:
require_file(ALL_BUILDINGS_OSM_FILE, 'OSM buildings (step 01)')

print('Reading the OSM building layer ...', flush=True)
t0 = time.perf_counter()
osm = gpd.read_file(ALL_BUILDINGS_OSM_FILE, layer='buildings')
print(f'  ok  {len(osm):,} rows x {len(osm.columns)} columns  '
      f'[{time.perf_counter() - t0:,.0f}s]')
require_non_empty(osm, 'osm buildings')
require_crs(osm, TARGET_CRS, 'osm buildings')
require_unique(osm, 'bld_id', 'osm buildings')
require_cols(osm, ['bld_id', 'building', 'building_levels', 'height', 'name',
                   'addr_street', 'addr_housenumber'], 'osm buildings')
if not osm.geometry.is_valid.all():
    raise AssertionError(f'{int((~osm.geometry.is_valid).sum()):,} invalid OSM geometries '
                         '- the intersection below would fail on them')
osm['area_m2'] = osm.geometry.area

# --- how much of each OSM footprint does ALKIS cover? -------------------------
# Candidate pairs by bounding box, then the exact intersection area per pair,
# summed per OSM footprint. Two ALKIS buildings overlapping each other would
# count the same ground twice, so the share is capped at 1.
print('Measuring ALKIS coverage of every OSM footprint (~30 s) ...', flush=True)
t0 = time.perf_counter()
pairs = gpd.sjoin(osm[['geometry']], slim[['geometry']], how='inner', predicate='intersects')
_li = osm.index.get_indexer(pairs.index)
_ri = slim.index.get_indexer(pairs['index_right'])
_inter = shapely.area(shapely.intersection(osm.geometry.values[_li],
                                           slim.geometry.values[_ri]))
_covered = pd.Series(_inter).groupby(_li).sum()
osm['alkis_coverage'] = (
    _covered.reindex(range(len(osm))).fillna(0.0).to_numpy() / osm['area_m2'].to_numpy()
).clip(max=1.0)
print(f'  ok  {len(pairs):,} intersecting pairs; '
      f'{len(_covered):,} of {len(osm):,} OSM footprints touch an ALKIS building  '
      f'[{time.perf_counter() - t0:,.0f}s]')

print()
print('  ..  ALKIS coverage of the OSM footprints - the threshold should sit in the valley:')
_bins = [-np.inf, 0.0, 0.05, 0.10, 0.20, 0.50, 1.0]
_names = ['exactly 0', '0 - 0.05', '0.05 - 0.10', '0.10 - 0.20', '0.20 - 0.50', '0.50 - 1.00']
_h = pd.cut(osm['alkis_coverage'], _bins, labels=_names).value_counts().sort_index()
for k, v in _h.items():
    mark = '  <- below threshold' if k in _names[:3] else ''
    print(f'        {k:<12} {v:>9,}   ({100 * v / len(osm):>5.2f} %){mark}')
print(f'        threshold: coverage < {OSM_GAP_MAX_ALKIS_COVERAGE}')

# --- the gap -----------------------------------------------------------------
is_gap = osm['alkis_coverage'] < OSM_GAP_MAX_ALKIS_COVERAGE
excluded = osm['building'].isin(OSM_GAP_EXCLUDE_BUILDING_TAGS)
gap = osm[is_gap & ~excluded].copy()
_share = 100 * len(gap) / len(osm)
print()
print(f'  ..  gap: {len(gap):,} OSM footprints ALKIS does not have '
      f'({_share:.2f} % of OSM, {gap["area_m2"].sum() / 1e6:.2f} km2, '
      f'median {gap["area_m2"].median():.1f} m2)')
print(f'  ..  excluded {int((is_gap & excluded).sum()):,} tagged '
      f'building={sorted(OSM_GAP_EXCLUDE_BUILDING_TAGS)}')
_lo, _hi = OSM_GAP_EXPECTED_SHARE_PCT
if not (_lo <= _share <= _hi):
    print(f'  !!  {_share:.1f} % is outside the expected {_lo}-{_hi} % band - '
          'check the two layers share a CRS and that step 01 ran on the same PBF')
else:
    print(f'  ok  within the expected {_lo}-{_hi} % band')

# Coverage-boundary effect: where LoD2 has no tile, ALKIS cannot have a record,
# so those gaps are a missing tile set rather than missing buildings.
if LOD2_REGION_TILES_FILE.exists():
    _tiles = gpd.read_file(LOD2_REGION_TILES_FILE).to_crs(TARGET_CRS)
    _tile_union = shapely.union_all(_tiles.geometry.values)
    _no_tile = ~gap.geometry.representative_point().within(_tile_union)
    print(f'  ..  {int(_no_tile.sum()):,} of them lie outside every LoD2 tile '
          '(no ALKIS coverage there at all, not a missing record)')

print()
print('  ..  what OSM says they are (building=*):')
_bt = gap['building'].value_counts()
for tag, c in _bt.head(15).items():
    print(f'        {tag:<22} {c:>7,}   ({100 * c / len(gap):>5.2f} %)')
print(f'        ... {len(_bt) - 15} more tags')
print()
print('  ..  what OSM knows about them:')
for c in ('name', 'building_levels', 'height', 'addr_street'):
    print(f'        {c:<16} {100 * gap[c].notna().mean():>6.2f} % filled')

# --- administrative key from the nearest ALKIS building ---------------------
_nn = gpd.sjoin_nearest(gap[['geometry']], slim[['ags', 'city', 'geometry']],
                        how='left', max_distance=OSM_GAP_AGS_MAX_DISTANCE_M,
                        distance_col='_d')
_nn = _nn[~_nn.index.duplicated(keep='first')]     # equidistant ties
gap['ags'] = _nn['ags']
gap['city'] = _nn['city']
print()
print(f'  ..  ags/city from the nearest ALKIS building: '
      f'{gap["ags"].notna().sum():,} of {len(gap):,} matched within '
      f'{OSM_GAP_AGS_MAX_DISTANCE_M} m (median {_nn["_d"].median():.0f} m, '
      f'99 % within {_nn["_d"].quantile(0.99):.0f} m)')

# --- rows in the layer's own schema ------------------------------------------
_s = gap['addr_street'].fillna('').astype(str).str.strip()
_h = gap['addr_housenumber'].fillna('').astype(str).str.strip()
_addr = (_s + ' ' + _h).str.strip()
_height = pd.to_numeric(gap['height'].astype(str).str.extract(r'(\d+(?:[.,]\d+)?)')[0]
                        .str.replace(',', '.'), errors='coerce')
_levels = pd.to_numeric(gap['building_levels'], errors='coerce')
# height estimate, the previous pipeline's rule: tag, else levels x floor, else one floor
_est_h = (_height.where(_height > 0)
          .fillna(_levels.where(_levels > 0) * OSM_GAP_FLOOR_HEIGHT_M)
          .fillna(OSM_GAP_DEFAULT_FLOORS * OSM_GAP_FLOOR_HEIGHT_M))
_est_v = (gap['area_m2'] * _est_h).round(1)
print()
print(f'  ..  height estimate: {int((_height > 0).sum()):,} from the height tag, '
      f'{int((_height.isna() & (_levels > 0)).sum()):,} from building:levels x {OSM_GAP_FLOOR_HEIGHT_M} m, '
      f'{int((_height.isna() & ~(_levels > 0)).sum()):,} default {OSM_GAP_DEFAULT_FLOORS} floor(s) = '
      f'{OSM_GAP_DEFAULT_FLOORS * OSM_GAP_FLOOR_HEIGHT_M} m; estimated volume {_est_v.sum() / 1e6:.1f}M m3 over {len(gap):,} footprints')

fill = gpd.GeoDataFrame({
    'building_id':  gap['bld_id'].to_numpy(),
    'source':       'osm',
    'alkis_id':     None,
    'function':     OSM_GAP_FUNCTION_CODE,
    'label_en':     OSM_GAP_LABEL_EN,
    'activities':   None,
    'n_activities': 0,
    'name':         gap['name'].to_numpy(),
    'address':      _addr.where(_addr != '', None).to_numpy(),
    'city':         gap['city'].to_numpy(),
    'ags':          gap['ags'].to_numpy(),
    'area_m2':      gap['area_m2'].round(2).to_numpy(),
    'n_parts':      1,
    'is_multipart': shapely.get_num_geometries(gap.geometry.values) > 1,
    'osm_building': gap['building'].to_numpy(),
    'osm_levels':   _levels.to_numpy(),
    'osm_height_m': _height.to_numpy(),
    'height_top_max_m': _est_h.round(2).to_numpy(),
    'height_top_avg_m': _est_h.round(2).to_numpy(),
    'volume_3d_m3': _est_v.to_numpy(),
    'volume_old_m3': _est_v.to_numpy(),
    'volume_ratio': 1.0,
    'geometry':     gap.geometry.values,
}, crs=TARGET_CRS)

slim['building_id'] = slim['alkis_id']
slim['source'] = 'alkis'
slim['osm_building'] = None
slim['osm_levels'] = np.nan
slim['osm_height_m'] = np.nan

fill = fill.reindex(columns=slim.columns)      # NULL for everything only ALKIS has
n_alkis = len(slim)
slim = gpd.GeoDataFrame(pd.concat([slim, fill], ignore_index=True),
                        geometry='geometry', crs=TARGET_CRS)
# Flags that are bool on ALKIS rows and NULL on OSM rows. Nullable boolean rather
# than object, so the GeoPackage gets 1/0/NULL and not the strings 'True'/'False'.
for c in ('roof_manual_any', 'roof_fallback_any', 'is_multipart'):
    slim[c] = slim[c].astype('boolean')
slim['n_parts'] = slim['n_parts'].astype('int64')

require_unique(slim, 'building_id', 'combined layer')
print(f'  ok  {n_alkis:,} ALKIS + {len(fill):,} OSM = {len(slim):,} buildings, '
      f'{len(slim.columns)} columns')
print()
print('  ..  the filled rows, as they now sit in the layer:')
_show = ['building_id', 'source', 'function', 'osm_building', 'name', 'address',
         'city', 'area_m2', 'osm_levels', 'osm_height_m', 'height_top_max_m', 'volume_3d_m3', 'activities']
print(slim.loc[slim['source'] == 'osm', _show].head(6).to_string(index=False))


  ok  01_all_buildings_osm.gpkg (165.4 MB)
Reading the OSM building layer ...


  ok  517,196 rows x 23 columns  [5s]
  ok  osm buildings: 517,196 rows
  ok  osm buildings: CRS EPSG:25832
  ok  osm buildings.bld_id: unique and non-null (517,196)
  ok  osm buildings: has ['bld_id', 'building', 'building_levels', 'height', 'name', 'addr_street', 'addr_housenumber']


Measuring ALKIS coverage of every OSM footprint (~30 s) ...


  ok  876,826 intersecting pairs; 471,062 of 517,196 OSM footprints touch an ALKIS building  [37s]

  ..  ALKIS coverage of the OSM footprints - the threshold should sit in the valley:
        exactly 0       46,134   ( 8.92 %)  <- below threshold
        0 - 0.05         2,507   ( 0.48 %)  <- below threshold
        0.05 - 0.10      1,374   ( 0.27 %)  <- below threshold
        0.10 - 0.20      2,204   ( 0.43 %)
        0.20 - 0.50     16,145   ( 3.12 %)
        0.50 - 1.00    448,832   (86.78 %)
        threshold: coverage < 0.1

  ..  gap: 50,011 OSM footprints ALKIS does not have (9.67 % of OSM, 5.34 km2, median 35.5 m2)
  ..  excluded 4 tagged building=['no']
  ok  within the expected 5.0-15.0 % band


  ..  848 of them lie outside every LoD2 tile (no ALKIS coverage there at all, not a missing record)

  ..  what OSM says they are (building=*):
        yes                     35,083   (70.15 %)
        house                    3,039   ( 6.08 %)
        garage                   2,301   ( 4.60 %)
        shed                     1,930   ( 3.86 %)
        detached                 1,430   ( 2.86 %)
        hut                        903   ( 1.81 %)
        semidetached_house         732   ( 1.46 %)
        roof                       699   ( 1.40 %)
        apartments                 550   ( 1.10 %)
        residential                464   ( 0.93 %)
        carport                    370   ( 0.74 %)
        garages                    272   ( 0.54 %)
        service                    206   ( 0.41 %)
        allotment_house            198   ( 0.40 %)
        bungalow                   184   ( 0.37 %)
        ... 76 more tags

  ..  what OSM knows about them:
        name               1.79


  ..  ags/city from the nearest ALKIS building: 49,915 of 50,011 matched within 1000 m (median 8 m, 99 % within 402 m)

  ..  height estimate: 626 from the height tag, 2,601 from building:levels x 2.0 m, 46,783 default 1 floor(s) = 2.0 m; estimated volume 13.1M m3 over 50,011 footprints


  ok  combined layer.building_id: unique and non-null (919,327)
  ok  869,316 ALKIS + 50,011 OSM = 919,327 buildings, 27 columns

  ..  the filled rows, as they now sit in the layer:
 building_id source function osm_building                    name    address                city  area_m2  osm_levels  osm_height_m  height_top_max_m  volume_3d_m3 activities
way/22943903    osm      OSM          yes                     NaN        NaN Braunschweig, Stadt    73.30         NaN           NaN               2.0         146.6       None
way/23862345    osm      OSM   commercial Hannes Camper Helmstedt Am Lohen 6    Helmstedt, Stadt   318.54         NaN           NaN               2.0         637.1       None
way/24989328    osm      OSM          yes                     NaN        NaN    Helmstedt, Stadt    97.75         NaN           NaN               2.0         195.5       None
way/25046958    osm      OSM          yes                     NaN        NaN      Süpplingenburg    78.40         NaN

## 7. Where each POI sits, and the two drop lists

Two facts decide whether a building survives this notebook: **what ALKIS says it
is** — its `function` code — and **whether OSM knows of anything happening
there**. Both are computed here, before the export, so the map can show every
decision. The order is deliberate:

1. **List 1 first.** `ALKIS_DROP_ALWAYS`, 29 codes: structures with no usable
   inside — canopies, masts, silos, tanks, chimneys, wind turbines, solar
   arrays, every kind of tower, stands, the stadium pitch polygons, ruins,
   monuments — plus three technical shells (water containers, drainage pumping
   stations, hiking shelters) and the wind and water mills. Gone regardless of any POI. Their **polygons** are
   dropped; their **POIs are not**.
2. **Then every POI is placed on an actual building** — one list 1 does not
   remove. A POI inside such a building is placed there. A POI inside a dropped
   structure (the fuel point under the forecourt canopy) or outside every
   footprint (the café node a few metres off the wall) is placed on the
   **nearest actual building within `POI_SNAP_MAX_DISTANCE_M`** — but only if
   its use is *building-bound*: at least `POI_BUILDING_BOUND_MIN_INSIDE_SHARE`
   of that use's POIs region-wide sit inside some footprint. Restaurants,
   supermarkets, kindergartens, farms do; swimming pools, ruins, graveyards and
   information boards do not, and stay unplaced rather than landing on a
   neighbour's house.
3. **Then list 2.** `ALKIS_DROP_UNLESS_POI`, 19 codes: buildings that are mostly
   not destinations but sometimes are — **residential buildings above all**
   (ALKIS codes a block by its dominant use, so the corner restaurant and the
   doctor's practice are `Wohngebäude` too), then farm buildings, parking,
   supply, disposal, transport operations, greenhouses, barracks, chapels,
   mourning halls, huts, mines, forester's houses. Dropped **unless a
   POI or a site is on them**.

**The OSM gap fill gets the same treatment by its `building=*` tag.**
`OSM_DROP_ALWAYS` — roof, shed, garage, hut, carport, service boxes, tanks,
allotment huts, … — goes blind in step 1; `OSM_DROP_UNLESS_POI` — house,
apartments, detached, barns, greenhouses, construction, parking, … — goes in
step 3 unless a POI or site is on the footprint. The bare `yes` — 33,603
footprints, 70 % under 50 m² — is in that list too: it says nothing, so without
a POI or site on it the footprint is noise (decided 2026-09-11). Only the ones
with an activity tag (commercial, retail, school, …) stay on their own. The tag
is read from `osm_building`, lower-cased.

4. **Then the size floor, in every class.** Below `SIZE_FLOOR_M2` (100 m²) a
   building stays only if something speaks for it: a POI on it, a site around
   it, an activity tag on its OSM footprint (`OSM_TWIN_ACTIVITY_TAGS`; for the
   OSM gap rows their own tag), a name on that footprint, or an ALKIS name —
   the cadastre's own label, *Vereinsheim*, *Feuerwehr*, *Sportheim*, unless
   the name is a structure word such as *Silos*, *Gas* or *Pumpwerk*
   (`ALKIS_NAME_NOT_EVIDENCE`). A
   footprint tagged garage, shed, carport, roof or hut
   (`OSM_TWIN_STRUCTURE_TAGS`) goes in step 1 with the structures and a POI on
   it moves next door. The floor was decided 2026-09-11 for class 2000 alone —
   369,675 buildings with a median footprint of 29 m², garages by the hundred
   thousand — from the size profile: three quarters of OSM-confirmed garages
   under 54 m², three quarters of OSM-confirmed commercial buildings over 132 m²,
   POI-carrying 2000-coded buildings at a median of 210 m². On 2026-09-14 it was
   extended to every class, because 12,372 kept buildings under 100 m² (25 % by
   count, 1.1 % by volume, median 42 m²) were the garages, storage and utility
   rooms of businesses coded 2010, 2100, 3000 or 3200; the ones with evidence
   stay, the rest go.

5. **Then the evidence band.** Between the floor and
   `ALKIS_SIZE_FLOOR_EVIDENCE_M2` — 100 to 200 m² for class 2000 — the same
   evidence keeps a building: a POI or site, an activity tag or a name on the
   OSM footprint, or an ALKIS name. An ALKIS address was considered and
   rejected: it proves a mailbox, not an activity, and gives the LLM nothing to
   classify from. A bare `yes` twin or no OSM footprint at all, with none of
   those, is a two-storey premises nobody has described, and it goes. Decided
   2026-09-11 from the profile of the 19,517 such buildings above the floor:
   median height 7.2 m, so not garages, but 8.6 % of the layer's volume with no
   usable information; the band under 200 m² takes 12,598 of them and leaves the
   larger halls as generic workplaces.
6. **Then the land.** For a building of a class that carries noise in numbers
   (`ALKIS_LANDUSE_RULE_CLASSES` — the generic non-residential codes 2000,
   2100, 2010, 3000, 3200 and the residential-first mixed codes 1110, 1120,
   1130; class 2000 at or above the evidence floor, the rest at every size)
   that *still* has nothing — no POI, no site, no OSM or ALKIS name, and
   the OSM twin, if any, mute: bare `yes`, a residential or farm tag such as
   `house` or `barn`, or a structure tag — the ALKIS land-use parcel under its
   centre decides: `residential` or `agriculture` (`ALKIS_LANDUSE_DROP`; the
   German layer names are translated through `ALKIS_LANDUSE_LABELS_EN`) and
   it goes, a house or a barn coded business; commercial, industrial, public or
   infrastructure land and it stays as a generic workplace; unknown and it
   stays. Classes that name a specific use — schools, fire stations, churches —
   and the business-first mixed codes with housing are not in the rule: there
   the ALKIS code *is* the information, and together they held 127 such
   buildings. ALKIS alone decides — OSM's land use is carried as
   `osm_landuse` for information, and the parcel's coded kind as
   `alkis_landuse_detail` (education and science, power plant, campsite ...;
   `ALKIS_LANDUSE_DETAIL_EN`, information only). OSM land use draws one residential polygon around
   a whole village and would call most business parks residential. Measured
   2026-09-11 on 36,827 such buildings: 6,645 on residential land (class 2000
   median 268 m², 9 m tall; class 1120 median 152 m², 10.6 m; class 2100 median
   89 m², 5 m), 3,435 on farmland, 26,747 on business, public or
   infrastructure land.
7. **Then the twin-structure rule, in every class and at every size.** If
   *every* OSM footprint on an ALKIS building is a garage, garages, carport,
   shed, hut or roof, OSM is saying the whole thing is a structure, whatever
   ALKIS coded it as — a 150 m² garage row coded 2000, a shed block coded
   industrial. It goes unless a POI or site is on it; the 67 petrol stations
   whose canopy ALKIS coded as the station building keep their fuel POI and
   stay. Where the ALKIS polygon holds a house *and* its garages the rule does
   not fire, because not every footprint is a structure.

Both ALKIS lists are **function codes** and both OSM lists are **tag values**,
decided class by class with the user on 2026-09-10/11 from the per-code profile
in section 5. The activity map plays
no part in the drop. It is used for one hint only: a code present in the data
whose activities are home-only and which sits in neither list is flagged,
because in another region that is almost certainly a residential code that
belongs in list 2.

### What "a POI on the building" means, per shape

* **`point`**, **`footprint`** and **`unit`** — a node, an OSM building polygon
  carrying the POI tag, or a shop unit inside a mall, reduced to one point.
  Placed as described above: inside, or snapped. A unit lands in its mall's
  building by construction. `n_poi` counts the ones inside, `n_poi_snap` the ones snapped on.
* **`site`** — an area polygon: a school's grounds, a care home, a campus, a
  riding centre, a holiday park. It says "activity somewhere in here", not "in
  this shed". Taken literally it would rescue the 285 garden huts of 1–3 m² in
  one allotment colony and the bike sheds on every school site. So a site
  rescues only the **substantial** buildings inside it: representative point
  within the site polygon *and* footprint at least
  `POI_SITE_RESCUE_MIN_AREA_M2` — bigger than any single-family house, smaller
  than a care-home wing. What the threshold keeps at other values is printed
  below so it can be moved. `in_site` marks the containment.

`has_poi` is *n_poi > 0, or n_poi_snap > 0, or in a site and at least the
threshold*. `drop_reason` records what removed a building — `list1`,
`floor_structure`, `list2_no_poi`, `floor_no_evidence`, `band_no_evidence`,
`land_residential_or_farm`, `twin_structure_no_poi`, or NULL for kept — `kept` is its negation, and `rescued` marks
the list-2 buildings a POI saved, with `rescued_by` saying what did the saving:
a `poi` inside, a `snap`ped one, a `site` around it, or — for a building under
the floor or in the evidence band — the `osm_tag` of the OSM footprint, its
`osm_name`, or the `alkis_name` on the cadastre record.

This is a *presence* test, not the POI join. 04.5 does the join properly — sites
onto every contained building, footprints 1:1, nested units — and **must use the
same placement** (inside, else snap for building-bound uses within the same
radius), so that the building a POI saved here is the building it is assigned to
there.

**The OSM fill carries `function = 'OSM'`**, which is in neither ALKIS list, so
only the OSM tag lists touch it. Its footprints are actual buildings, so a POI
can be placed on them and can save a list-2 one.

Two guards. Every code in the two lists must exist in the codelist — a typo
would otherwise be a silent no-op, which is exactly the bug the old pipeline
shipped with — and the lists must not overlap. Any `51xxx` structure class
present in the data but in neither list is flagged, because on new data that is
almost certainly an omission.


In [8]:
# --- 7a. the lists must be typo-free and disjoint, and nothing obvious missing --
_known = set(codes['function'])
for _nm, _lst in (('ALKIS_DROP_ALWAYS', ALKIS_DROP_ALWAYS),
                  ('ALKIS_DROP_UNLESS_POI', ALKIS_DROP_UNLESS_POI)):
    _bad = sorted(set(_lst) - _known)
    if _bad:
        raise AssertionError(f'{_nm} names codes that are not in the codelist - '
                             f'a typo here is a silent no-op: {_bad}')
_both = sorted(set(ALKIS_DROP_ALWAYS) & set(ALKIS_DROP_UNLESS_POI))
if _both:
    raise AssertionError(f'codes in both drop lists: {_both}')
_present = set(slim['function'])
print(f'  ok  list 1: {len(ALKIS_DROP_ALWAYS)} codes, list 2: {len(ALKIS_DROP_UNLESS_POI)} codes; '
      f'all in the codelist, disjoint; {len(_present & set(ALKIS_DROP_ALWAYS))} + '
      f'{len(_present & set(ALKIS_DROP_UNLESS_POI))} of them occur in this region')

slim['aaa_class'] = slim['function'].str.split('_').str[0]
_in_lists = set(ALKIS_DROP_ALWAYS) | set(ALKIS_DROP_UNLESS_POI)
_unruled = [c for c in sorted(_present - _in_lists)
            if c != OSM_GAP_FUNCTION_CODE and not c.startswith('31001_')]
if _unruled:
    print(f'  !!  {len(_unruled)} structure classes (not 31001) are in neither list and will be '
          f'KEPT - almost certainly an omission: {_unruled}')
else:
    print('  ok  every non-31001 structure class present is in a drop list')

# The activity map's only role in the drop: a hint. A code whose activities are
# home-only and which is in neither list is, in all likelihood, a residential
# code this region does not use but another one will.
_home_only_codes = set(acts.loc[
    acts['activities'].fillna('').str.split(ALKIS_ACTIVITY_SEP)
    .map(lambda xs: frozenset(x.strip() for x in xs if x.strip()))
    .map(lambda a: bool(a) and a <= ALKIS_HOME_ONLY_ACTIVITIES), 'function'])
_hint = sorted((_home_only_codes & _present) - _in_lists)
if _hint:
    print(f'  !!  {len(_hint)} codes present have home-only activities but are in neither list - '
          f'probably belong in list 2: {_hint}')
else:
    print('  ok  every code present with home-only activities is in a drop list')

# The OSM gap fill has no AdV code; its lists key on the OSM building=* tag.
_osm_tag = slim['osm_building'].fillna('').astype(str).str.strip().str.lower()
_is_osm = slim['source'] == 'osm'
_oboth = sorted(set(OSM_DROP_ALWAYS) & set(OSM_DROP_UNLESS_POI))
if _oboth:
    raise AssertionError(f'OSM tags in both drop lists: {_oboth}')
_otags = _osm_tag[_is_osm].value_counts()
_okept = _otags[~_otags.index.isin(set(OSM_DROP_ALWAYS) | set(OSM_DROP_UNLESS_POI))]
print(f'  ok  OSM tag lists: {len(OSM_DROP_ALWAYS)} + {len(OSM_DROP_UNLESS_POI)} tags, disjoint; '
      f'{int(_otags[_otags.index.isin(OSM_DROP_ALWAYS)].sum()):,} + '
      f'{int(_otags[_otags.index.isin(OSM_DROP_UNLESS_POI)].sum()):,} of the {int(_otags.sum()):,} '
      f'gap-fill footprints fall under them; {int(_okept.sum()):,} keep their tag: '
      + ', '.join(f'{t} {n:,}' for t, n in _okept.head(8).items())
      + (f', ... {len(_okept) - 8} more tags' if len(_okept) > 8 else ''))

_always = (slim['function'].isin(ALKIS_DROP_ALWAYS)            # list 1: these polygons go first
           | (_is_osm & _osm_tag.isin(OSM_DROP_ALWAYS)))
_unless = (slim['function'].isin(ALKIS_DROP_UNLESS_POI)        # list 2: decided by the POIs below
           | (_is_osm & _osm_tag.isin(OSM_DROP_UNLESS_POI)))

# --- the size floor: what does the OSM footprint on top of an ALKIS building say?
# The OSM building whose representative point falls inside the ALKIS polygon is
# its twin; where several do, the most frequent tag wins. Computed for every
# ALKIS row - it is a useful column in QGIS - and used by the floor rule.
_op = osm[['building', 'geometry']].dropna(subset=['building']).copy()
_op['geometry'] = _op.geometry.representative_point()
_twj = gpd.sjoin(_op, slim.loc[~_is_osm, ['building_id', 'geometry']], how='inner', predicate='within')
_twj['building'] = _twj['building'].astype(str).str.strip().str.lower()
_tw = (_twj.groupby(['building_id', 'building']).size().reset_index(name='n')
       .sort_values(['building_id', 'n'], ascending=[True, False]).drop_duplicates('building_id'))
slim['osm_twin_tag'] = slim['building_id'].map(_tw.set_index('building_id')['building'])
# ... and whether EVERY footprint on it is a structure (garage, shed, roof, ...):
# then OSM says the whole building is one, whatever ALKIS coded it as.
_pure = _twj.groupby('building_id')['building'].agg(lambda s: bool(s.isin(OSM_TWIN_STRUCTURE_TAGS).all()))
slim['osm_twin_all_structure'] = slim['building_id'].map(_pure).fillna(False).astype(bool)
# ... and its NAME, whatever the tag. A named building=yes is not a POI, but its
# name is information the LLM should see: 'Rechenzentrum', 'Alte Schule'.
_on = osm[['name', 'geometry']].dropna(subset=['name']).copy()
_on['geometry'] = _on.geometry.representative_point()
_tn = gpd.sjoin(_on, slim.loc[~_is_osm, ['building_id', 'geometry']], how='inner', predicate='within')
_tn = _tn.groupby('building_id')['name'].agg(lambda s: '; '.join(dict.fromkeys(s.astype(str))))
slim['osm_twin_name'] = slim['building_id'].map(_tn)
print()
print(f'  ..  OSM twin footprint found for {int(slim["osm_twin_tag"].notna().sum()):,} of {int((~_is_osm).sum()):,} ALKIS buildings; '
      f'{int(slim["osm_twin_name"].notna().sum()):,} of those carry a name')

# --- the size floor, EVERY class: below SIZE_FLOOR_M2 a building needs evidence ------------
_below = slim['area_m2'] < SIZE_FLOOR_M2
_twin = slim['osm_twin_tag']                                        # ALKIS rows: the OSM footprint on them
_tag_any = _twin.fillna(slim['osm_building'])                       # OSM gap rows: their own building tag
_floor_hard = _below & _twin.isin(OSM_TWIN_STRUCTURE_TAGS) & ~_always   # goes with the structures (list 1 keeps its own label)
# EVIDENCE that something happens inside - the same set for the floor, the band and the land rule
_ev_tag = _tag_any.isin(OSM_TWIN_ACTIVITY_TAGS)                     # OSM calls the footprint commercial, school, church ...
_ev_osm_name = slim['osm_twin_name'].notna()                        # the OSM footprint is named
_nm = slim['name'].astype('string').str.strip().str.lower()
_nm_structure = _nm.isin(ALKIS_NAME_NOT_EVIDENCE) | _nm.str.split().str[0].isin(ALKIS_NAME_NOT_EVIDENCE)
_ev_alkis_name = slim['name'].notna() & ~_is_osm & ~_nm_structure.fillna(False).astype(bool)   # the cadastre named it: Vereinsheim, Feuerwehr - not Silos, Gas, WC
print(f'  ..  ALKIS names: {int((slim["name"].notna() & ~_is_osm).sum()):,} buildings carry one; '
      f'{int((slim["name"].notna() & ~_is_osm & _nm_structure.fillna(False).astype(bool)).sum()):,} are structure words (ALKIS_NAME_NOT_EVIDENCE) and do not count as evidence: '
      + ', '.join(f'{k} {v}' for k, v in slim.loc[_nm_structure.fillna(False).astype(bool) & ~_is_osm, 'name'].value_counts().head(8).items()))
_ev_any = _ev_tag | _ev_osm_name | _ev_alkis_name
_floor_keep = _below & ~_floor_hard & _ev_any                        # stays: something speaks for it
_floor_soft = _below & ~_floor_hard & ~_floor_keep                  # nothing yet: POIs and sites decide
print()
print(f'  ..  size floor {SIZE_FLOOR_M2:.0f} m2 on every class: {int((_below & ~_always).sum()):,} buildings below it outside list 1 - '
      f'{int(_floor_hard.sum()):,} have a garage/shed twin and go with the structures, '
      f'{int(_floor_keep.sum()):,} have an activity tag or a name and stay '
      f'({int((_floor_keep & _ev_tag).sum()):,} tag, {int((_floor_keep & ~_ev_tag & _ev_osm_name).sum()):,} OSM name, '
      f'{int((_floor_keep & ~_ev_tag & ~_ev_osm_name).sum()):,} ALKIS name), '
      f'{int(_floor_soft.sum()):,} are decided by the POIs and sites')
print('      below the floor outside list 1, by class: '
      + ', '.join(f'{k} {v:,}' for k, v in slim.loc[_below & ~_always, 'function'].value_counts().head(8).items()))
# the evidence band: at or above the floor, under the class's evidence floor
_floor2 = pd.Series(np.nan, index=slim.index, dtype=float)
for _code, _f in ALKIS_SIZE_FLOOR_EVIDENCE_M2.items():
    _floor2[slim['function'] == _code] = _f
_band = ~_below & (slim['area_m2'] < _floor2)
_band_soft = _band & ~_ev_any                                       # only a POI or site can still keep it
for _code, _f in ALKIS_SIZE_FLOOR_EVIDENCE_M2.items():
    _m = (slim['function'] == _code) & _band
    print(f'  ..  evidence band {SIZE_FLOOR_M2:.0f}-{_f:.0f} m2 on {_code}: {int(_m.sum()):,} buildings - '
          f'{int((_m & _ev_tag).sum()):,} have an activity twin, {int((_m & ~_ev_tag & _ev_osm_name).sum()):,} a named twin, '
          f'{int((_m & ~_ev_tag & ~_ev_osm_name & _ev_alkis_name).sum()):,} an ALKIS name, '
          f'{int((_m & _band_soft).sum()):,} have none and are decided by the POIs')
# --- the land under every building: ALKIS Landnutzung (decides) and OSM landuse (information)
require_file(ALKIS_LANDUSE_FILE, 'ALKIS Landnutzung (statewide)')
require_file(LANDUSE_OSM_FILE, 'OSM land use (step 01)')
print()
print('  ..  clipping the statewide ALKIS land use to the region (~15 s) ...', flush=True)
t0 = time.perf_counter()
_bbox = tuple(slim.total_bounds)
_lu_parts, _unmapped = [], []
for _lyr in [l[0] for l in pyogrio.list_layers(ALKIS_LANDUSE_FILE)]:
    if _lyr not in ALKIS_LANDUSE_LABELS_EN:
        raise KeyError(f'ALKIS land-use layer {_lyr!r} has no English label in config.ALKIS_LANDUSE_LABELS_EN')
    _spec = ALKIS_LANDUSE_DETAIL_EN.get(_lyr)                          # (attribute, {code: label}) or None
    if _spec and _spec[0] not in pyogrio.read_info(ALKIS_LANDUSE_FILE, layer=_lyr)['fields']:
        raise KeyError(f'config.ALKIS_LANDUSE_DETAIL_EN expects column {_spec[0]!r} in layer {_lyr!r}, which has no such column')
    _g = pyogrio.read_dataframe(ALKIS_LANDUSE_FILE, layer=_lyr, columns=['uuid'] + ([_spec[0]] if _spec else []), bbox=_bbox)
    _g['alkis_landuse'] = ALKIS_LANDUSE_LABELS_EN[_lyr]                # English only reaches the layer
    _g['alkis_landuse_detail'] = None
    if _spec:                                                          # the parcel's coded kind, decoded - never the raw code
        _code = _g[_spec[0]].astype('string').str.strip()
        _det = _code.map(_spec[1])
        _g['alkis_landuse_detail'] = _det.astype(object).where(_det.notna(), None)
        _unk = _code[_code.notna() & _det.isna()].value_counts()
        if len(_unk):
            _unmapped.append(f'{_lyr}: ' + ', '.join(f'{k} x{v:,}' for k, v in _unk.head(5).items()))
    _lu_parts.append(_g[['uuid', 'alkis_landuse', 'alkis_landuse_detail', 'geometry']])
alkis_lu = gpd.GeoDataFrame(pd.concat(_lu_parts, ignore_index=True), crs=TARGET_CRS)
require_crs(alkis_lu, TARGET_CRS, 'ALKIS land use')
_rep = slim[['building_id', 'geometry']].copy()
_rep['geometry'] = _rep.geometry.representative_point()
_h = gpd.sjoin(_rep, alkis_lu[['alkis_landuse', 'alkis_landuse_detail', 'geometry']], how='inner', predicate='within')
_h = _h[~_h.index.duplicated(keep='first')]
slim['alkis_landuse'] = _h['alkis_landuse'].reindex(_rep.index).to_numpy()
slim['alkis_landuse_detail'] = _h['alkis_landuse_detail'].reindex(_rep.index).to_numpy()
print(f'  ok  {len(alkis_lu):,} land-use polygons in the region; '
      f'{int(slim["alkis_landuse"].notna().sum()):,} of {len(slim):,} buildings have a class under their centre  '
      f'[{time.perf_counter() - t0:,.0f}s]')
print('      buildings by land use (top 8): ' + ', '.join(f'{k} {v:,}' for k, v in slim['alkis_landuse'].value_counts().head(8).items()))
print(f'  ..  the parcel kind (alkis_landuse_detail) is decoded under {int(slim["alkis_landuse_detail"].notna().sum()):,} buildings - top 10: '
      + ', '.join(f'{k} {v:,}' for k, v in slim['alkis_landuse_detail'].value_counts().head(10).items()))
if _unmapped:
    print('  !!  codes present in the region but not in config.ALKIS_LANDUSE_DETAIL_EN (left NULL, not invented): ' + ' | '.join(_unmapped))
osm_lu = gpd.read_file(LANDUSE_OSM_FILE, layer='landuse')[['landuse', 'area_m2', 'geometry']]
_ho = gpd.sjoin(_rep, osm_lu, how='inner', predicate='within').sort_values('area_m2')   # smallest polygon wins
_ho = _ho[~_ho.index.duplicated(keep='first')]
slim['osm_landuse'] = _ho['landuse'].reindex(_rep.index).to_numpy()
print(f'  ..  OSM land use under {int(slim["osm_landuse"].notna().sum()):,} buildings - information only: '
      + ', '.join(f'{k} {v:,}' for k, v in slim['osm_landuse'].value_counts().head(6).items()))

# the land-use rule: a generic business class, above any floor it has, nothing describes it, the land says living or farming.
# A twin is MUTE when it says nothing about activity: absent, bare 'yes', a residential or farm tag, or a structure tag.
_twin_mute = _twin.isna() | _twin.isin(OSM_DROP_UNLESS_POI) | _twin.isin(OSM_TWIN_STRUCTURE_TAGS)
_undescribed = (slim['function'].isin(ALKIS_LANDUSE_RULE_CLASSES) & ~_below & ~_band
                & _twin_mute & ~_ev_osm_name & ~_ev_alkis_name)
_land_soft = _undescribed & slim['alkis_landuse'].isin(ALKIS_LANDUSE_DROP)
for _code in sorted(ALKIS_LANDUSE_RULE_CLASSES):
    _m = (slim['function'] == _code) & _undescribed
    _fl = ALKIS_SIZE_FLOOR_EVIDENCE_M2.get(_code)
    _scope = f'at/above {_fl:.0f} m2' if _fl else 'at every size'
    print(f'  ..  land-use rule on {_code} {_scope} with a mute or no twin and no name (OSM or ALKIS): '
          f'{int(_m.sum()):,} buildings - by land: '
          + ', '.join(f'{k} {v:,}' for k, v in slim.loc[_m, 'alkis_landuse'].fillna('<none>').value_counts().head(8).items()))
    print(f'      {int((_m & _land_soft).sum()):,} stand on {"/".join(sorted(ALKIS_LANDUSE_DROP))} and go unless a POI or site is on them')

_struct_soft = (slim['osm_twin_all_structure'] & ~_floor_hard
                & ~slim['function'].isin(ALKIS_DROP_ALWAYS))       # every footprint a structure: POIs decide, any class, any size
_always = _always | _floor_hard
print(f'  ..  twin-structure rule: {int(_struct_soft.sum()):,} buildings outside list 1 and the floor have ONLY garage/shed/roof '
      f'footprints on them - kept only if a POI or site is on them. By class: '
      + ', '.join(f'{k} {v:,}' for k, v in slim.loc[_struct_soft, 'function'].value_counts().head(6).items()))

# --- 7b. points and footprints: where does each POI sit? ----------------------
require_file(ALL_POIS_FILE, 'OSM POIs (step 01)')
pois = gpd.read_file(ALL_POIS_FILE, layer='pois')
require_non_empty(pois, 'pois')
require_crs(pois, TARGET_CRS, 'pois')
require_unique(pois, 'poi_id', 'pois')
require_cols(pois, ['poi_id', 'poi_role', 'poi_use', 'name', 'area_m2'], 'pois')
print()
print('  ok  ' + f'{len(pois):,} POIs: '
      + ', '.join(f'{k} {v:,}' for k, v in pois['poi_role'].value_counts().items()))

_pf = pois.loc[pois['poi_role'].isin(['point', 'footprint', 'unit']),
               ['poi_id', 'poi_role', 'poi_use', 'name', 'geometry']].copy()
_pf['geometry'] = _pf.geometry.representative_point()

# Step 1 - containment against EVERY footprint, structures included. It tells
# which POIs sit inside a list-1 structure (the fuel point under the forecourt
# canopy), and it gives the per-use share of POIs that sit on some built thing at
# all - which is what separates building-bound uses from outdoor ones.
_all = gpd.sjoin(_pf, slim[['building_id', 'geometry']].assign(_r1=_always.to_numpy()),
                 how='left', predicate='within')
_all = _all[~_all.index.duplicated(keep='first')]     # a point on a shared wall: one building
_in_any = _all['building_id'].notna()
_in_r1 = _all['_r1'].fillna(False).astype(bool)
_share = _in_any.groupby(_all['poi_use']).mean()
_cnt = _pf['poi_use'].value_counts()
_bound_uses = set(_share[_share >= POI_BUILDING_BOUND_MIN_INSIDE_SHARE].index)
_pf['bound'] = _pf['poi_use'].isin(_bound_uses)
print(f'  ..  {int(_in_any.sum()):,} of {len(_pf):,} point/footprint POIs sit inside some footprint: '
      f'{int((_in_any & ~_in_r1).sum()):,} inside an actual building, '
      f'{int(_in_r1.sum()):,} inside a list-1 structure; {int((~_in_any).sum()):,} inside none')
print(f'  ..  building-bound uses (>= {POI_BUILDING_BOUND_MIN_INSIDE_SHARE:.0%} of the use sits inside a footprint): '
      f'{len(_bound_uses)} of {_share.size} uses, {int(_pf["bound"].sum()):,} POIs')
_outdoor = _share[_share < POI_BUILDING_BOUND_MIN_INSIDE_SHARE].index
_outdoor = sorted(_outdoor, key=lambda u: -_cnt[u])[:10]
print('      outdoor uses, biggest: ' + ', '.join(f'{u} {_share[u]:.0%} of {_cnt[u]}' for u in _outdoor))

# Step 2 - placement on ACTUAL buildings, i.e. everything list 1 does not remove.
# Inside one -> placed there. Inside a structure or outside every footprint ->
# the nearest actual building within POI_SNAP_MAX_DISTANCE_M, building-bound
# uses only. Everything else stays unplaced.
_actual = slim.loc[~_always, ['building_id', 'function', 'source', 'geometry']]
poi_hits = gpd.sjoin(_pf, _actual, how='inner', predicate='within')
poi_hits = poi_hits[~poi_hits.index.duplicated(keep='first')]
poi_hits['placed_by'] = 'inside'
_stray = _pf[~_pf['poi_id'].isin(poi_hits['poi_id'])]
_snap = gpd.sjoin_nearest(_stray[_stray['bound']], _actual, how='inner',
                          max_distance=POI_SNAP_MAX_DISTANCE_M, distance_col='snap_m')
_snap = _snap[~_snap.index.duplicated(keep='first')]
_snap['placed_by'] = 'snap'
poi_hits = pd.concat([poi_hits, _snap])
_unplaced = _stray[~_stray['poi_id'].isin(_snap['poi_id'])]
_from_r1 = _all.index[_in_r1]
_from_out = _all.index[~_in_any]
_placed_ids = set(poi_hits['poi_id'])
print()
print(f'  ..  placed on an actual building: {len(poi_hits):,} of {len(_pf):,} POIs - '
      f'{int((poi_hits["placed_by"] == "inside").sum()):,} inside one, '
      f'{len(_snap):,} snapped to the nearest within {POI_SNAP_MAX_DISTANCE_M} m')
print(f'      of the {len(_from_r1):,} inside a list-1 structure: '
      f'{int(_pf.loc[_from_r1, "poi_id"].isin(_placed_ids).sum()):,} placed on the actual building next to '
      f'(or under) it, {int((~_pf.loc[_from_r1, "poi_id"].isin(_placed_ids)).sum()):,} not')
print(f'      of the {len(_from_out):,} outside every footprint: '
      f'{int(_pf.loc[_from_out, "poi_id"].isin(_placed_ids).sum()):,} snapped, '
      f'{int((~_pf.loc[_from_out, "poi_id"].isin(_placed_ids)).sum()):,} not')
print(f'  ..  unplaced: {len(_unplaced):,} - {int((~_unplaced["bound"]).sum()):,} of outdoor uses, '
      f'{int(_unplaced["bound"].sum()):,} building-bound with nothing within {POI_SNAP_MAX_DISTANCE_M} m')
print('      snapped, by use: ' + ', '.join(f'{k} {v}' for k, v in _snap['poi_use'].value_counts().head(10).items()))
print(f'      snap distance: median {_snap["snap_m"].median():.1f} m, 90 % within {_snap["snap_m"].quantile(0.9):.0f} m')

_per = (poi_hits.groupby(['building_id', 'placed_by']).size().unstack(fill_value=0)
        .reindex(columns=['inside', 'snap'], fill_value=0))
slim['n_poi'] = slim['building_id'].map(_per['inside']).fillna(0).astype('int64')
slim['n_poi_snap'] = slim['building_id'].map(_per['snap']).fillna(0).astype('int64')
for _src, _g in slim.groupby('source'):
    print(f'        {_src:<6} {int(((_g["n_poi"] + _g["n_poi_snap"]) > 0).sum()):>7,} buildings carry a POI  '
          f'({int(_g["n_poi"].sum()):,} inside, {int(_g["n_poi_snap"].sum()):,} snapped)')

# --- 7c. sites: only the substantial buildings inside an area POI -------------
_sites = pois.loc[pois['poi_role'] == 'site', ['poi_id', 'poi_use', 'name', 'geometry']]
_rep = slim[['building_id', 'geometry']].copy()
_rep['geometry'] = _rep.geometry.representative_point()
site_hits = gpd.sjoin(_rep, _sites, how='inner', predicate='within')
site_hits = site_hits[~site_hits.index.duplicated(keep='first')]   # nested sites: one is enough
slim['in_site'] = slim['building_id'].isin(site_hits['building_id'])
_own = (slim['n_poi'] + slim['n_poi_snap']) > 0
slim['has_poi'] = _own | (slim['in_site'] & (slim['area_m2'] >= POI_SITE_RESCUE_MIN_AREA_M2))
print()
print(f'  ..  {len(_sites):,} site polygons contain {int(slim["in_site"].sum()):,} buildings; '
      f'{int((slim["in_site"] & ~_own).sum()):,} of them carry no POI of their own')
_cand = slim['in_site'] & ~_own & (_unless | _floor_soft | _band_soft | _land_soft | _struct_soft)
print(f'  ..  {int(_cand.sum()):,} of those are list-2, below-floor or structure-twin buildings - what a site ALONE rescues, '
      'by minimum footprint:')
for _t in (0, 30, 50, 100, 200, 500):
    _m = _cand & (slim['area_m2'] >= _t)
    _mark = '  <- POI_SITE_RESCUE_MIN_AREA_M2' if _t == POI_SITE_RESCUE_MIN_AREA_M2 else ''
    print(f'        >= {_t:>3} m2   {int(_m.sum()):>5,} buildings   '
          f'{slim.loc[_m, "volume_3d_m3"].sum() / 1e6:>5.2f}M m3{_mark}')
_big = _cand & (slim['area_m2'] >= POI_SITE_RESCUE_MIN_AREA_M2)
_bs = site_hits.loc[site_hits['building_id'].isin(slim.loc[_big, 'building_id']), 'poi_use']
print('      the sites doing the rescuing: '
      + ', '.join(f'{k} {v}' for k, v in _bs.value_counts().head(8).items()))

# --- 7d. the two lists, list 1 first ------------------------------------------
slim['drop_reason'] = np.select(
    [_always & ~_floor_hard, _floor_hard, _unless & ~slim['has_poi'], _floor_soft & ~slim['has_poi'],
     _band_soft & ~slim['has_poi'], _land_soft & ~slim['has_poi'], _struct_soft & ~slim['has_poi']],
    ['list1', 'floor_structure', 'list2_no_poi', 'floor_no_evidence', 'band_no_evidence', 'land_residential_or_farm',
     'twin_structure_no_poi'],
    default=None)
slim['kept'] = slim['drop_reason'].isna()
_saved_by_poi = (_unless | _floor_soft | _band_soft | _land_soft | _struct_soft) & slim['has_poi']
_band_kept = _band & ~_band_soft                          # kept by tag or a name
_size_kept = _floor_keep | _band_kept
slim['rescued'] = _saved_by_poi | _size_kept
slim['rescued_by'] = np.select(
    [_saved_by_poi & (slim['n_poi'] > 0), _saved_by_poi & (slim['n_poi_snap'] > 0), _saved_by_poi,
     _size_kept & _ev_tag, _size_kept & _ev_osm_name, _size_kept & _ev_alkis_name],
    ['poi', 'snap', 'site', 'osm_tag', 'osm_name', 'alkis_name'],
    default=None)
# a building can be spoken for and still go (an address in the band, but every
# footprint on it a garage): the flags describe kept buildings only
slim.loc[~slim['kept'], 'rescued'] = False
slim.loc[~slim['kept'], 'rescued_by'] = None

print()
print('  ..  what the lists decide (applied and reported in section 9):')
_t = (slim.groupby(slim['drop_reason'].fillna('kept'))
      .agg(buildings=('building_id', 'size'),
           volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 1)),
           pois_on_them=('n_poi', 'sum')))
print(_t.reindex(['list1', 'floor_structure', 'list2_no_poi', 'floor_no_evidence', 'band_no_evidence', 'land_residential_or_farm',
                  'twin_structure_no_poi', 'kept']).to_string())
print(f'  ..  rescued: {int(slim["rescued"].sum()):,} list-2 or below-floor buildings kept - '
      f'{int((slim["rescued_by"] == "poi").sum()):,} by a POI inside, '
      f'{int((slim["rescued_by"] == "snap").sum()):,} by a snapped POI, '
      f'{int((slim["rescued_by"] == "site").sum()):,} by a site polygon around them, '
      f'{int((slim["rescued_by"] == "osm_tag").sum()):,} by an activity tag on the OSM twin, '
      f'{int((slim["rescued_by"] == "osm_name").sum()):,} by a named twin')
_sv = poi_hits[(poi_hits['placed_by'] == 'snap')
               & poi_hits['building_id'].isin(slim.loc[slim['rescued_by'] == 'snap', 'building_id'])]
print(f'      snapped POIs that saved a building ({len(_sv):,}), by use: '
      + ', '.join(f'{k} {v}' for k, v in _sv['poi_use'].value_counts().head(10).items()))


  ok  list 1: 29 codes, list 2: 19 codes; all in the codelist, disjoint; 29 + 19 of them occur in this region


  ok  every non-31001 structure class present is in a drop list
  ok  every code present with home-only activities is in a drop list


  ok  OSM tag lists: 34 + 27 tags, disjoint; 7,157 + 42,267 of the 50,011 gap-fill footprints fall under them; 587 keep their tag: commercial 139, industrial 128, office 47, kindergarten 47, warehouse 45, retail 43, school 38, fire_station 13, ... 22 more tags



  ..  OSM twin footprint found for 432,306 of 869,316 ALKIS buildings; 9,089 of those carry a name


  ..  ALKIS names: 5,029 buildings carry one; 324 are structure words (ALKIS_NAME_NOT_EVIDENCE) and do not count as evidence: Silos 54, Trafo 33, Gas 27, (ev.) 21, Ter. 18, Wbh 17, Gülle 14, Pumpwerk 12

  ..  size floor 100 m2 on every class: 542,199 buildings below it outside list 1 - 24,625 have a garage/shed twin and go with the structures, 1,946 have an activity tag or a name and stay (786 tag, 813 OSM name, 347 ALKIS name), 618,657 are decided by the POIs and sites
      below the floor outside list 1, by class: 31001_2000 341,125, 31001_1000 141,938, OSM 30,195, 31001_2720 7,714, 31001_2500 7,127, 31001_2100 4,354, 31001_2010 3,454, 31001_3200 949


...


  ..  placed on an actual building: 22,652 of 25,750 POIs - 20,590 inside one, 2,062 snapped to the nearest within 50 m
      of the 298 inside a list-1 structure: 241 placed on the actual building next to (or under) it, 57 not
      of the 4,862 outside every footprint: 1,821 snapped, 3,041 not
  ..  unplaced: 3,098 - 2,752 of outdoor uses, 346 building-bound with nothing within 50 m
      snapped, by use: kindergarten 158, industrial 154, commercial 117, fuel 96, school 74, fast_food 69, restaurant 60, sports_centre 57, farm 42, retail 41
      snap distance: median 3.8 m, 90 % within 24 m


        alkis   16,693 buildings carry a POI  (19,804 inside, 1,923 snapped)
        osm        764 buildings carry a POI  (786 inside, 139 snapped)



  ..  2,093 site polygons contain 45,376 buildings; 40,111 of them carry no POI of their own
  ..  27,928 of those are list-2, below-floor or structure-twin buildings - what a site ALONE rescues, by minimum footprint:
        >=   0 m2   27,928 buildings   17.21M m3
        >=  30 m2   15,439 buildings   16.47M m3
        >=  50 m2   11,222 buildings   15.78M m3
        >= 100 m2   5,688 buildings   13.78M m3
        >= 200 m2   2,526 buildings   11.28M m3  <- POI_SITE_RESCUE_MIN_AREA_M2
        >= 500 m2     875 buildings    8.12M m3
      the sites doing the rescuing: commercial 858, industrial 848, social_facility 115, retail 100, wastewater_plant 85, factory 46, research_institute 45, school 36



  ..  what the lists decide (applied and reported in section 9):
                          buildings  volume_Mm3  pois_on_them
drop_reason                                                  
list1                        107923        19.2             0
floor_structure               24625         2.9             0
list2_no_poi                 394278       324.2             0
floor_no_evidence            327628        35.4             0
band_no_evidence              16312        11.4             0
land_residential_or_farm       8532        19.7             0
twin_structure_no_poi           243         0.4             0
kept                          39786       277.2         20590
  ..  rescued: 10,938 list-2 or below-floor buildings kept - 6,115 by a POI inside, 854 by a snapped POI, 2,526 by a site polygon around them, 756 by an activity tag on the OSM twin, 504 by a named twin


      snapped POIs that saved a building (878), by use: industrial 100, commercial 46, fuel 43, farm 40, fast_food 34, kindergarten 32, sports_centre 31, kiosk 23, restaurant 20, cafe 17


## 8. The class-by-class review export

Every building in the layer, labelled — the 869,316 from ALKIS **and the OSM
footprints section 6 added**, **including the ones section 9 drops** — so the
decision can be made by looking rather than from counts alone.

GeoPackage rather than shapefile: `label_en` and `activities` are long strings
and shapefile truncates field names to 10 characters.

This is the **full** layer, dropped rows included, so the drop can be judged by
looking. Section 9 writes a second file, **`04_buildings_kept.gpkg`**, with only
the buildings that survive — the layer to look at when the question is "what
is left?" rather than "what went?". Same columns, same style, class counts
recomputed for what is left.

### Stepping through the classes

A **`.qml` style file** is written beside the layer, so QGIS loads it already
categorised by `function` with all 89 classes in the legend — the 88 AdV codes
and the `OSM` fill. Each legend entry
can be ticked on and off individually — that is the class-by-class review,
without typing a single filter expression.

Colours are grouped by what the code family means, so the map is readable before
you touch anything:

| family | colour |
|---|---|
| `31001_1xxx` residential | blues |
| `31001_2xxx` business, commerce | oranges and reds |
| `31001_3xxx` public purposes | greens |
| `51xxx` other structures | greys |
| `OSM` footprints ALKIS does not have | magenta |

Legend entries read `31001_1000 · residential buildings · 327,264` so the label
carries the code, the meaning and the size without a lookup.

Eleven extra columns make manual filtering easy where you do want it:

* **`class_label`** — the same `code · meaning · count` string, if you would
  rather categorise on that than on the bare code
* **`aaa_class`** — `31001` or `51009` etc., for splitting buildings from
  structures in one filter
* **`kept`** — what section 9 removes, as one flag. Styling by it shows the
  drop directly.
* **`drop_reason`** — *what* removed it: `list1` (outright), `floor_structure`
  (under the floor with a garage/shed twin), `list2_no_poi` (list 2,
  no POI or site on it), `floor_no_evidence` (any class under 100 m² with no
  POI, site, activity tag or name), `band_no_evidence` (class 2000 between 100
  and 200 m² with none of those), `land_residential_or_farm` (class 2000 above the evidence floor, nothing
  describes it, and the ALKIS parcel under it is residential or agricultural),
  `twin_structure_no_poi` (every OSM footprint on it is a garage, shed or roof,
  nothing on it), NULL for kept
* **`alkis_landuse_detail`** — the parcel's coded kind where ALKIS has one,
  decoded to English: *education and science*, *health and spa*, *power
  plant*, *campsite*, *allotment gardens*. The codes come from the parcel's
  `funktion` or `art` column and their meaning from the AdV codelists in the
  GDI-DE registry (`https://registry.gdi-de.org/codelist/de.adv-online.gid/`,
  the code is the last path segment: `.../AX_Funktion_FlaecheBesonderer-
  FunktionalerPraegung/1120` is *Bildung und Wissenschaft*). Information for
  the LLM, no rule reads it. What it buys: `DENIAL01000050Yi` in Braunschweig
  is *Buildings for public purposes*, 787 m², five storeys, an `office`
  footprint in OSM and nothing else — a town hall as far as the class knows;
  the parcel says *education and science*, and it stands beside the
  Lebensmittel- und Veterinärinstitut. Filled on 7,493 of the kept buildings,
  on 821 of the 8,216 that carry no POI, site or name.
* **`alkis_landuse`** and **`osm_landuse`** — the land-use class under the
  building's centre from ALKIS Landnutzung and from OSM. ALKIS decides in the
  rule above; OSM is information
* **`osm_twin_tag`** and **`osm_twin_name`** — the `building=*` tag and the
  name(s) of the OSM footprint(s) on top of the ALKIS building, where they exist.
  The name is carried whatever the tag: information for the LLM, never a keep or
  drop signal
* **`rescued`** — true where a POI saved a building rule 2 or 3 would have
  dropped: the residential block with the restaurant, the parking garage with
  the supermarket, the care-home wing coded residential
* **`rescued_by`** — `poi` when a point or footprint POI sits inside the
  building, `snap` when one was placed on it from a dropped canopy or from just
  outside the wall, `site` when only a site polygon around it did the saving
  (and the building is at least `POI_SITE_RESCUE_MIN_AREA_M2`)
* **`n_poi`** — point/footprint POIs inside the footprint, **`n_poi_snap`** —
  POIs snapped onto it from within `POI_SNAP_MAX_DISTANCE_M`, and
  **`in_site`** — whether the building lies inside a site polygon at all
* **`source`** — `alkis` or `osm`, and **`osm_building`** — the OSM `building=*`
  tag on the filled rows, for splitting the magenta class by what OSM thinks
  each footprint is.

### Worth looking at specifically

1. **`rescued = true`** — the buildings that are here only because of a POI.
   With `rescued_by = 'poi'` each should show a shop, a practice, a restaurant
   or a stable inside a residential-coded, farm or parking polygon; one that
   does not is a misplaced POI. With `rescued_by = 'snap'` the POI sits a few
   metres away, under a canopy or just outside the wall; the building should
   still be the obvious host. With `rescued_by = 'site'` each should be a
   main building of the facility around it — a care-home wing, a riding hall, a
   campus building — not a shed; a shed here means the threshold is too low.
2. **`drop_reason = 'list1'`** — the 27 classes dropped outright, the canopies
   above all: 94,571 of them, median 10 m² and 3.9 m. Tick classes on and off in
   the legend to see each one.
3. **`function = '31001_2000' AND area_m2 < 50`** — the shed question, still
   open. 369,675 buildings are tagged "business or commerce" with a median of
   28.7 m².
4. **`drop_reason = 'list2_no_poi'`** — the list-2 buildings nothing saved. With
   `function = '31001_1000'` these should be housing and nothing else; with
   `function = '31001_2720'` barns and stables.
5. **`source = 'osm'`** — the fill. Every magenta footprint should sit in a
   visible hole in the ALKIS layer; one drawn on top of an ALKIS building means
   the coverage threshold let a duplicate through.


In [9]:
EXPERIMENTAL_DIR.mkdir(parents=True, exist_ok=True)

# Columns that make the review easy without writing filter expressions.
# `aaa_class`, `kept`, `drop_reason`, `rescued` and `n_poi` come from section 7.
_n = slim['function'].value_counts()
slim['class_label'] = (
    slim['function'] + ' \u00b7 ' + slim['label_en'].fillna('?')
    + ' \u00b7 ' + slim['function'].map(_n).map('{:,}'.format))

inspect_cols = ['building_id', 'source', 'alkis_id', 'function', 'aaa_class',
                'class_label', 'label_en', 'activities', 'n_activities',
                'kept', 'drop_reason', 'rescued', 'rescued_by', 'n_poi', 'n_poi_snap', 'in_site',
                'area_m2', 'volume_3d_m3', 'volume_old_m3', 'volume_ratio',
                'height_top_max_m', 'roof_shape', 'name', 'address', 'city',
                'n_parts', 'osm_building', 'osm_twin_tag', 'osm_twin_name', 'osm_twin_all_structure', 'alkis_landuse', 'alkis_landuse_detail', 'osm_landuse',
                'osm_levels', 'osm_height_m', 'geometry']
insp = slim[inspect_cols]

if LABELLED_INSPECT_FILE.exists():
    try:
        LABELLED_INSPECT_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{LABELLED_INSPECT_FILE.name} is locked - close it in QGIS and rerun '
            f'this cell. Original error: {e}'
        ) from None

print(f'Writing {len(insp):,} rows x {len(insp.columns)} columns to '
      f'{LABELLED_INSPECT_FILE.name} ...', flush=True)
t0 = time.perf_counter()
insp.to_file(LABELLED_INSPECT_FILE, layer='buildings', driver='GPKG')
print(f'  ok  {LABELLED_INSPECT_FILE.stat().st_size / 1e6:,.1f} MB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

chk = gpd.read_file(LABELLED_INSPECT_FILE, layer='buildings', rows=4)
print(f'  ok  read back: CRS {chk.crs}, {len(chk.columns)} columns')
print(chk[['function', 'label_en', 'activities', 'kept', 'drop_reason', 'n_poi',
           'area_m2', 'height_top_max_m']].to_string(index=False))
_chk_osm = gpd.read_file(LABELLED_INSPECT_FILE, layer='buildings',
                         where="source = 'osm'", rows=3)
print(f'  ok  read back {len(_chk_osm)} OSM rows by `source` filter:')
print(_chk_osm[['building_id', 'function', 'osm_building', 'kept', 'area_m2']]
      .to_string(index=False))

Writing 919,327 rows x 36 columns to 04_buildings_labelled.gpkg ...


  ok  535.0 MB  [17s]
  ok  read back: CRS EPSG:25832, 36 columns
  function                           label_en    activities  kept  drop_reason  n_poi  area_m2  height_top_max_m
51002_1250                               mast          work False        list1      0     4.00             3.500
51002_1250                               mast          work False        list1      0     4.00             3.500
31001_1000              residential buildings   home;meetup False list2_no_poi      0   115.68             7.404
31001_2000 Buildings for business or commerce work;business  True          NaN      0   212.80             4.377


  ok  read back 3 OSM rows by `source` filter:
 building_id function osm_building  kept  area_m2
way/22943903      OSM          yes False    73.30
way/23862345      OSM   commercial  True   318.54
way/24989328      OSM          yes  True    97.75


In [10]:
# --- write a QGIS style so the layer opens already categorised --------------
# Hand-written QML rather than exported from QGIS, because QGIS is not scriptable
# from here. Uses the <prop k=.../> form, which QGIS 3.x still accepts, and is
# belt-and-braces: if a QGIS version refuses the file, `class_label` still gives
# a three-click path to the same categorised view.
#
# Colours carry meaning rather than being arbitrary: the family a code belongs to
# decides the hue, so the map is readable before any styling is touched.
_FAMILY_RAMPS = {
    '1': [(31, 120, 180), (66, 146, 198), (107, 174, 214), (158, 202, 225),
          (198, 219, 239), (222, 235, 247)],                      # residential: blues
    '2': [(227, 74, 51), (239, 101, 72), (252, 141, 89), (253, 187, 132),
          (253, 212, 158), (254, 232, 200)],                      # business: oranges/reds
    '3': [(35, 132, 67), (65, 171, 93), (120, 198, 121), (173, 221, 142),
          (217, 240, 163), (247, 252, 185)],                      # public: greens
}
_GREYS = [(82, 82, 82), (115, 115, 115), (150, 150, 150), (189, 189, 189),
          (217, 217, 217)]
_OSM_FILL_COLOUR = (231, 41, 138)                                 # OSM fill: magenta


def _colour(code, i):
    if code == OSM_GAP_FUNCTION_CODE:
        return _OSM_FILL_COLOUR
    aaa, gfk = code.split('_', 1)
    if aaa != '31001':
        return _GREYS[i % len(_GREYS)]
    ramp = _FAMILY_RAMPS.get(gfk[0], _GREYS)
    return ramp[i % len(ramp)]


def _esc(t):
    return (str(t).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


def _write_qml(layer, path):
    # Categorised by `function`, biggest classes first, legend entries
    # 'code · meaning · count' from the layer's OWN counts - so the kept-only
    # file shows what is left, not what went in.
    _n = layer['function'].value_counts()
    _lab = layer.drop_duplicates('function').set_index('function')['label_en'].fillna('?')
    order = _n.index.tolist()
    cats, syms, seen = [], [], {}
    for n, code in enumerate(order):
        if code == OSM_GAP_FUNCTION_CODE:
            fam = 'osm'
        elif code.startswith('31001'):
            fam = code.split('_', 1)[1][0]
        else:
            fam = 'x'
        seen[fam] = seen.get(fam, 0)
        r, g, b = _colour(code, seen[fam])
        seen[fam] += 1
        label = f'{code} · {_lab[code]} · {_n[code]:,}'
        cats.append(f'   <category value="{_esc(code)}" symbol="{n}" '
                    f'label="{_esc(label)}" render="true"/>')
        syms.append(
            f'   <symbol type="fill" name="{n}" alpha="1" clip_to_extent="1" force_rhr="0">\n'
            f'    <layer class="SimpleFill" enabled="1" pass="0" locked="0">\n'
            f'     <prop k="color" v="{r},{g},{b},255"/>\n'
            f'     <prop k="style" v="solid"/>\n'
            f'     <prop k="outline_color" v="35,35,35,180"/>\n'
            f'     <prop k="outline_style" v="solid"/>\n'
            f'     <prop k="outline_width" v="0.04"/>\n'
            f'     <prop k="outline_width_unit" v="MM"/>\n'
            f'     <prop k="joinstyle" v="bevel"/>\n'
            f'    </layer>\n'
            f'   </symbol>')
    qml = (
        "<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>\n"
        '<qgis version="3.28.0" styleCategories="Symbology">\n'
        ' <renderer-v2 type="categorizedSymbol" attr="function" forceraster="0"\n'
        '              symbollevels="0" enableorderby="0" referencescale="-1">\n'
        '  <categories>\n' + '\n'.join(cats) + '\n  </categories>\n'
        '  <symbols>\n' + '\n'.join(syms) + '\n  </symbols>\n'
        ' </renderer-v2>\n'
        ' <blendMode>0</blendMode>\n'
        '</qgis>\n'
    )
    path.write_text(qml, encoding='utf-8')
    # Parse it back - a malformed QML fails silently in QGIS, which is worse
    # than an error here.
    import xml.etree.ElementTree as _ET
    _root = _ET.fromstring(qml)
    _ncat = len(_root.findall('.//category'))
    _nsym = len(_root.findall('.//symbol'))
    if _ncat != len(order) or _nsym != len(order):
        raise AssertionError(f'QML has {_ncat} categories and {_nsym} symbols, '
                             f'expected {len(order)} of each')
    print(f'  ok  {path.name}: {_ncat} categories, valid XML, '
          f'{path.stat().st_size / 1e3:,.0f} KB')
    return order


order = _write_qml(slim, LABELLED_INSPECT_FILE.with_suffix('.qml'))
labels = slim.drop_duplicates('function').set_index('function')['class_label']
print()
print('  ..  in QGIS: add 04_buildings_labelled.gpkg - the .qml loads with it,')
print('      and every class appears in the legend with its own tick box;')
print(f'      the OSM fill is the magenta entry "{labels.get(OSM_GAP_FUNCTION_CODE, OSM_GAP_FUNCTION_CODE)}".')
print()
print('  ..  the legend, biggest first:')
for code in order[:12]:
    print(f'        {labels.get(code, code)}')
print(f'        ... and {len(order) - 12} more')


  ok  04_buildings_labelled.qml: 89 categories, valid XML, 51 KB



  ..  in QGIS: add 04_buildings_labelled.gpkg - the .qml loads with it,
      and every class appears in the legend with its own tick box;
      the OSM fill is the magenta entry "OSM · Building mapped in OSM, no ALKIS record · 50,011".

  ..  the legend, biggest first:
        31001_2000 · Buildings for business or commerce · 369,675
        31001_1000 · residential buildings · 327,264
        51009_1610 · canopy · 94,571
        OSM · Building mapped in OSM, no ALKIS record · 50,011
        31001_2720 · Agricultural and forestry business building · 18,850
        31001_2100 · Commercial and industrial buildings · 11,849
        31001_2010 · Buildings for trade and services · 8,631
        31001_2500 · Buildings for supply · 7,806
        31001_1120 · Residential buildings with trade and services · 5,793
        31001_1210 · Agricultural and forestry residential building · 4,197
        31001_3000 · Buildings for public purposes · 3,301
        51003_1201 · silo · 1,784
        ... a

## 9. Drop what is not a place of activity

A capacity model places demand that happens *at* a building — work, shopping,
errands, education, leisure. Section 7 marked two kinds of row that host none of
it; this section reports each list and removes what it selected.

### List 1 — structures dropped outright

`ALKIS_DROP_ALWAYS` in `config.py`, one line of reasoning per code. The bulk is
the 94,571 canopies — forecourt roofs, carports, bus shelters — followed by
silos, masts, solar arrays, wind turbines, tanks, chimneys, every kind of tower,
stands, the stadium pitch polygons, ruins and monuments, plus water containers,
drainage pumping stations, hiking shelters and the wind and water mills.

**No POI can save these — but the POIs on them are not lost.** 145 POIs sit
under canopies — Aral, Esso, Shell and Jet forecourts, a Lidl, two Sparkasse
branches, three pharmacies — because the mapper put the point under the roof.
Every one has a real building within 21 m, typically about a metre. Section 7
drops the canopy and places the POI on that building, where it counts for list 2
like any other. The count printed below says how many were re-homed and how many
stayed unplaced because their use is an outdoor one.

### List 2 — buildings dropped unless a POI or a site is on them

`ALKIS_DROP_UNLESS_POI`, 19 codes, all real buildings that are mostly not
destinations but sometimes are.

**Residential buildings are the case that matters.** `31001_1000` is 327,264
buildings and 44 % of the region's volume; nearly all of it is housing, which a
capacity model does not place demand on. But ALKIS codes a building by its
dominant use, so the corner restaurant, the hairdresser, the doctor's practice,
the care home and the student hall are `Wohngebäude` too. Where step 01 placed a
point or footprint POI on such a building it stays, `rescued_by = 'poi'`; where a
building-bound POI was snapped onto it from a few metres away, `'snap'`; where it
is a substantial building inside a site polygon, `'site'`. The same correction
the previous pipeline made, with the size guard section 7 explains.

The rest of the list follows the same logic: farm buildings, where 95 of 18,850
hold a riding stable, a farm shop or a café; parking garages, where an Aldi, a
KiK and two gyms occupy the ground floor of a few; supply, disposal and
transport-operations buildings, greenhouses, barracks, chapels, mourning halls,
huts, mines and forester's houses. The POI keeps the building; the rest
go.

### The size floor, in every class

`SIZE_FLOOR_M2` puts a 100 m² floor under every class. Below it the OSM
footprint on top of the building decides first — garage, shed, carport, roof,
hut and the building goes with the structures. Then evidence: an activity tag
on the footprint (`rescued_by = 'osm_tag'`), a name on it (`'osm_name'`), the
cadastre's own name (`'alkis_name'`: Vereinsheim, Sportheim, Feuerwehr — but
not a structure word like Silos or Gas, `ALKIS_NAME_NOT_EVIDENCE`), or a
POI or site. With none of those a 40 m² *trade and services* building on
commercial land is the storage room of the business next door, and it goes.
Until 2026-09-14 the floor applied to class 2000 only; 12,372 small buildings
of the other classes had stayed, a quarter of the layer by count and 1.1 % by
volume.

### The evidence band

Above the floor, class 2000 still held 19,517 buildings with nothing at all —
a bare `yes` twin or no OSM footprint, no POI, no site, no name, no address —
8.6 % of the layer's volume. They are not garages (median height 7.2 m); they
are premises nobody has described. Between 100 and 200 m²
(`ALKIS_SIZE_FLOOR_EVIDENCE_M2`) such a building now goes, while the larger
ones stay as generic workplaces. Any evidence keeps a building in the band:
a POI or site, an activity tag or a name on the twin, or an ALKIS name —
each recorded in `rescued_by`. An ALKIS address was tried as a fourth signal and
dropped again: it says a mailbox exists, not what happens inside, and the LLM
could not classify from it.

### The land under the leftovers

The eight classes in the rule still held 36,827 buildings that nothing
(January OSM snapshot; the September rerun moved every figure here by under 1 %)
describes — 10,763 class-2000 halls above the evidence floor, 10,079 class-2100
buildings, 6,692 class 2010, 5,112 class 1120 and the rest — each with no POI,
no site, no twin name and at most a mute twin (`yes`, `house`, `apartments`,
`barn`, `garages`). The ALKIS land-use parcel under each decides: 6,645 stand
on residential land — two-storey houses coded business or *residential with
trade*, and workshops of 89 m² behind houses coded industrial — 3,435 stand on
farmland and are barns and machinery halls coded business; both go unless a
POI or site is on them, which saves 526. The 26,747 on commercial, industrial,
public or infrastructure land stay as generic workplaces, as does anything on
unclassified land. ALKIS alone decides; OSM's land use is on the layer for
information — it calls 3,024 of the halls on ALKIS industrial land and 6,887
on commercial-services land residential.

### The twin-structure rule

The size floor only looks at class 2000 under 100 m², so a 150 m² garage row
coded 2000, or a shed block coded industrial, sailed through with a
`garages` twin — 2,676 kept buildings on the first run. Now a building whose
OSM footprints are *all* garage, garages, carport, shed, hut or roof goes in
every class and at every size, unless a POI or site is on it. The 121 that
carry one — 67 petrol stations whose canopy ALKIS coded as the station, car
washes, a compost site — stay.

### The OSM fill is judged by its tag

The rows section 6 added have no AdV code, but they have OSM's `building=*`
tag, and the same two lists exist for it. `OSM_DROP_ALWAYS` — roof, shed,
garage, hut, carport, service boxes, tanks, allotment huts — goes blind, and the
15 fuel and car-wash POIs under `roof` move to the building next to them like
the canopy POIs. `OSM_DROP_UNLESS_POI` — house, apartments, detached, barns,
greenhouses, construction, parking — goes unless a POI or site is on the
footprint. The 33,603 footprints tagged just `yes` say nothing, so they too go
unless a POI or site is on them. What an OSM tag means in activity terms for the kept ones is
still a reference table nobody has written.

### What this costs, decided deliberately

**`meetup` is a redistribution target.** The original pipeline maps MiD
`meetup → Leisure`, and dropping `31001_1000` removes about 280M m³ of it. So
visiting-friends trips have nowhere to land, and Leisure demand falls entirely
on pubs, sports halls and cinemas.

Accepted: home visits are out of scope for a capacity model. Worth revisiting
here if the Leisure totals later look implausibly concentrated on venues.


In [11]:
def _mm3(v):
    return v.sum() / 1e6


# --- 9a. list 1: dropped outright, with the POIs that sat on them -------------
print('LIST 1 - dropped outright (ALKIS_DROP_ALWAYS), biggest first:')
_r1_cnt = _all.loc[_in_r1].groupby('building_id').size()
_c = slim[slim['drop_reason'] == 'list1'].copy()
_c['_p'] = _c['building_id'].map(_r1_cnt).fillna(0).astype('int64')
_a = (_c.groupby(['function', 'label_en'])
      .agg(n=('building_id', 'size'), vol=('volume_3d_m3', _mm3), pois=('_p', 'sum'))
      .reset_index().sort_values('n', ascending=False))
for _, r in _a.iterrows():
    print(f'  {r["function"]:<12} {r["label_en"][:38]:<38} {int(r["n"]):>7,} bld  '
          f'{r["vol"]:>6.2f}M m3  {int(r["pois"]):>4} POIs inside')
_absent = sorted(set(ALKIS_DROP_ALWAYS) - set(_a['function']))
print(f'  {len(_a)} classes, {int(_a["n"].sum()):,} buildings, {_a["vol"].sum():.1f}M m3'
      + (f'; {len(_absent)} listed codes do not occur in this region: {_absent}' if _absent else ''))
_o1 = _c[_c['source'] == 'osm']
if len(_o1):
    _ot = (_o1.groupby(_osm_tag[_o1.index]).agg(n=('building_id', 'size'), pois=('_p', 'sum'))
           .sort_values('n', ascending=False))
    print(f'  ..  of which OSM footprints, by building=* tag ({len(_o1):,}, {int(_ot["pois"].sum())} POIs inside):')
    for _tag, _r in _ot.head(12).iterrows():
        print(f'        {_tag:<20} {int(_r["n"]):>6,}  {int(_r["pois"]):>3} POIs inside')
    if len(_ot) > 12:
        print(f'        ... {len(_ot) - 12} more tags, {int(_ot["n"].iloc[12:].sum()):,} footprints')
_under = _pf.loc[_in_r1.reindex(_pf.index).fillna(False).to_numpy()]
_rehomed = _under['poi_id'].isin(poi_hits['poi_id'])
print(f'  ..  {len(_under):,} POIs sat inside these: {int(_rehomed.sum()):,} placed on the actual '
      f'building next to (or under) them (section 7), {int((~_rehomed).sum()):,} left unplaced. By use: '
      + ', '.join(f'{k} {v}' for k, v in _under['poi_use'].value_counts().head(8).items()))
print('      unplaced ones, by use: '
      + ', '.join(f'{k} {v}' for k, v in _under.loc[~_rehomed, 'poi_use'].value_counts().head(8).items()))

# --- 9b. list 2: dropped unless a POI or site is on them ----------------------
print()
print('LIST 2 - dropped unless a POI or site is on them (ALKIS_DROP_UNLESS_POI), biggest first:')
_l2 = slim[_unless]
for _code in _l2['function'].value_counts().index:
    _g = _l2[_l2['function'] == _code]
    _k = _g[_g['kept']]
    print(f'  {_code:<12} {_g["label_en"].iloc[0][:38]:<38} {len(_g):>7,} bld -> '
          f'{len(_k):>5,} kept ({int((_k["rescued_by"] == "poi").sum()):,} inside, '
          f'{int((_k["rescued_by"] == "snap").sum()):,} snapped, '
          f'{int((_k["rescued_by"] == "site").sum()):,} site), {len(_g) - len(_k):,} dropped '
          f'({_mm3(_g["volume_3d_m3"]) - _mm3(_k["volume_3d_m3"]):.2f}M m3)')
    _u = poi_hits.loc[poi_hits['building_id'].isin(_k['building_id']), 'poi_use'].value_counts().head(6)
    if len(_u):
        print(f'  {"":<12} POIs on the kept ones: ' + ', '.join(f'{a} {b}' for a, b in _u.items()))
_o2 = _l2[_l2['source'] == 'osm']
if len(_o2):
    print(f'  ..  of which OSM footprints, by building=* tag ({len(_o2):,}):')
    for _tag in _o2.groupby(_osm_tag[_o2.index]).size().sort_values(ascending=False).index:
        _g = _o2[_osm_tag[_o2.index] == _tag]
        _k = _g[_g['kept']]
        print(f'        {_tag:<20} {len(_g):>6,} -> {len(_k):>4,} kept '
              f'({int((_k["rescued_by"] == "poi").sum())} inside, {int((_k["rescued_by"] == "snap").sum())} snapped, '
              f'{int((_k["rescued_by"] == "site").sum())} site), {len(_g) - len(_k):,} dropped')
_absent2 = sorted(set(ALKIS_DROP_UNLESS_POI) - set(_l2['function']))
print(f'  {_l2["function"].nunique()} classes, {len(_l2):,} buildings -> {int(_l2["kept"].sum()):,} kept, '
      f'{int((~_l2["kept"]).sum()):,} dropped ({_mm3(_l2.loc[~_l2["kept"], "volume_3d_m3"]):.1f}M m3)'
      + (f'; {len(_absent2)} listed codes do not occur in this region: {_absent2}' if _absent2 else ''))
_sr = _l2[_l2['rescued_by'] == 'site']
_su = site_hits.loc[site_hits['building_id'].isin(_sr['building_id']), 'poi_use'].value_counts()
print(f'  ..  kept by a site polygon alone ({len(_sr):,}, median {_sr["area_m2"].median():,.0f} m2): '
      + ', '.join(f'{a} {b}' for a, b in _su.head(10).items()))


# --- 9c. the size floor, every class, by what the OSM footprint says -----------------
print()
_b = slim[_below & (slim['drop_reason'] != 'list1')].copy()
print(f'SIZE FLOOR - every class under {SIZE_FLOOR_M2:.0f} m2 ({len(_b):,} buildings outside list 1; {int((~_below).sum()):,} at or above the floor):')
_tg = _b['osm_twin_tag'].fillna(_b['osm_building'])
_cat = np.select(
    [_tg.isna(), _tg.isin(OSM_TWIN_STRUCTURE_TAGS), _tg.isin(OSM_TWIN_ACTIVITY_TAGS), _tg == 'yes'],
    ['no OSM footprint', 'garage/shed twin', 'activity twin', 'bare yes twin'], default='other twin')
_bb = _b.assign(_cat=_cat)
_r = (_bb.groupby('_cat')
      .agg(n=('building_id', 'size'), kept=('kept', 'sum'),
           by_poi=('rescued_by', lambda s: int(s.isin(['poi', 'snap', 'site']).sum())),
           by_tag=('rescued_by', lambda s: int((s == 'osm_tag').sum())),
           by_name=('rescued_by', lambda s: int(s.isin(['osm_name', 'alkis_name']).sum())))
      .sort_values('n', ascending=False))
_gv = _bb.loc[~_bb['kept']].groupby('_cat')['volume_3d_m3'].sum() / 1e6
for _c, r in _r.iterrows():
    print(f'  {_c:<18} {int(r["n"]):>7,} bld -> {int(r["kept"]):>6,} kept ({int(r["by_poi"]):,} by a POI or site, '
          f'{int(r["by_tag"]):,} by the tag, {int(r["by_name"]):,} by a name), {int(r["n"] - r["kept"]):,} dropped ({_gv.get(_c, 0.0):.2f}M m3)')
_gone = _b[~_b['kept']]
print(f'  ..  {len(_gone):,} dropped below the floor ({_mm3(_gone["volume_3d_m3"]):.2f}M m3, median {_gone["area_m2"].median():.0f} m2), by class: '
      + ', '.join(f'{k} {v:,}' for k, v in _gone['label_en'].value_counts().head(8).items()))
print('  ..  kept by an ALKIS name alone: '
      + ', '.join(f'{k} {v}' for k, v in _b.loc[_b['rescued_by'] == 'alkis_name', 'name'].value_counts().head(10).items()))
print('  ..  the POIs that kept small buildings, by use: '
      + ', '.join(f'{k} {v}' for k, v in poi_hits.loc[poi_hits['building_id'].isin(_b.loc[_b['rescued_by'].isin(['poi', 'snap']), 'building_id']), 'poi_use'].value_counts().head(10).items()))

# --- 9d. the evidence band ------------------------------------------------------------
print()
for _code, _f2 in ALKIS_SIZE_FLOOR_EVIDENCE_M2.items():
    _m = (slim['function'] == _code) & _band
    _bd = slim[_m]
    print(f'EVIDENCE BAND - {_code} between {SIZE_FLOOR_M2:.0f} and {_f2:.0f} m2 ({len(_bd):,} buildings):')
    _how = _bd['rescued_by'].fillna('dropped')
    for _k, _v in _how.value_counts().items():
        print(f'  {_k:<10} {int(_v):>7,}')
    _gone = _bd[~_bd['kept']]
    print(f'  ..  {len(_gone):,} dropped ({_mm3(_gone["volume_3d_m3"]):.1f}M m3); twin: '
          + ', '.join(f'{k} {v:,}' for k, v in _gone['osm_twin_tag'].fillna('<none>').value_counts().head(4).items())
          + f'; median height {_gone["height_top_max_m"].median():.1f} m')

# --- 9e. the land under the undescribed halls -----------------------------------------
print()
_ud = slim[_undescribed]
print(f'LAND-USE RULE - classes {", ".join(sorted(ALKIS_LANDUSE_RULE_CLASSES))}, mute or no twin, no name ({len(_ud):,} buildings):')
for _code in sorted(ALKIS_LANDUSE_RULE_CLASSES):
    _c = _ud[_ud['function'] == _code]
    print(f'  {_code}: {len(_c):,} buildings -> {int(_c["kept"].sum()):,} kept, {int((~_c["kept"]).sum()):,} dropped '
          f'({_mm3(_c.loc[~_c["kept"], "volume_3d_m3"]):.1f}M m3); twin: '
          + ', '.join(f'{k} {v:,}' for k, v in _c['osm_twin_tag'].fillna('<none>').value_counts().head(6).items()))
_r = (_ud.groupby(_ud['alkis_landuse'].fillna('<none>'))
      .agg(n=('building_id', 'size'), kept=('kept', 'sum'), vol=('volume_3d_m3', _mm3),
           osm_res=('osm_landuse', lambda s: int((s == 'residential').sum())))
      .sort_values('n', ascending=False))
for _k, r in _r.head(10).iterrows():
    print(f'  {_k:<34} {int(r["n"]):>6,} bld -> {int(r["kept"]):>5,} kept, {int(r["n"] - r["kept"]):,} dropped  '
          f'({r["vol"]:.1f}M m3; OSM calls {int(r["osm_res"]):,} of them residential)')
_g = _ud[~_ud['kept']]
print(f'  ..  {len(_g):,} dropped ({_mm3(_g["volume_3d_m3"]):.1f}M m3), median {_g["area_m2"].median():.0f} m2 and {_g["height_top_max_m"].median():.1f} m tall; '
      f'{int(_ud["kept"].sum()):,} stay as generic workplaces')

# --- 9f. the twin-structure rule -----------------------------------------------------
print()
_st = slim[_struct_soft]
print(f'TWIN-STRUCTURE RULE - every OSM footprint on the building is a garage/shed/roof ({len(_st):,} buildings outside list 1 and the floor):')
_r = (_st.groupby(['function', 'label_en'])
      .agg(n=('building_id', 'size'), kept=('kept', 'sum'), vol=('volume_3d_m3', _mm3))
      .reset_index().sort_values('n', ascending=False))
for _, r in _r.head(10).iterrows():
    print(f'  {r["function"]:<12} {r["label_en"][:38]:<38} {int(r["n"]):>6,} bld -> {int(r["kept"]):>4,} kept by a POI or site, {int(r["n"] - r["kept"]):,} dropped')
print(f'  {len(_r)} classes, {len(_st):,} buildings -> {int(_st["kept"].sum()):,} kept, {int((~_st["kept"]).sum()):,} dropped '
      f'({_mm3(_st.loc[~_st["kept"], "volume_3d_m3"]):.1f}M m3); by twin: '
      + ', '.join(f'{k} {v:,}' for k, v in _st['osm_twin_tag'].value_counts().head(6).items()))
print('  ..  the POIs that kept some, by use: '
      + ', '.join(f'{k} {v}' for k, v in poi_hits.loc[poi_hits['building_id'].isin(_st.loc[_st['kept'], 'building_id']), 'poi_use'].value_counts().head(8).items()))

# --- 9g. apply ----------------------------------------------------------------
n_before = len(slim)
v_before = slim['volume_3d_m3'].sum()
enriched = (slim[slim['kept']]
            .drop(columns=['has_poi', 'n_poi', 'n_poi_snap', 'in_site', 'drop_reason', 'kept'])
            .reset_index(drop=True))
require_unique(enriched, 'building_id', 'enriched buildings')
print()
print(f'  ..  buildings : {n_before:,} -> {len(enriched):,} '
      f'({n_before - len(enriched):,} dropped, '
      f'{100 * (n_before - len(enriched)) / n_before:.2f} %)')
print(f'  ..  volume    : {v_before / 1e6:,.1f}M -> '
      f'{enriched["volume_3d_m3"].sum() / 1e6:,.1f}M m3 '
      f'({100 * (1 - enriched["volume_3d_m3"].sum() / v_before):.2f} % removed)')
for _src in ('alkis', 'osm'):
    print(f'        {_src:<6} {int((slim["source"] == _src).sum()):>8,} -> '
          f'{int((enriched["source"] == _src).sum()):>8,}')
print(f'  ..  rescued   : {int(enriched["rescued"].sum()):,} buildings are here only because something spoke for them - '
      f'{int((enriched["rescued_by"] == "poi").sum()):,} a POI inside, '
      f'{int((enriched["rescued_by"] == "snap").sum()):,} a snapped POI, '
      f'{int((enriched["rescued_by"] == "site").sum()):,} a site polygon, '
      f'{int((enriched["rescued_by"] == "osm_tag").sum()):,} the activity tag of the OSM twin, '
      f'{int((enriched["rescued_by"] == "osm_name").sum()):,} a named twin, '
      f'{int((enriched["rescued_by"] == "alkis_name").sum()):,} an ALKIS name '
      '(`rescued`, `rescued_by` carried forward)')

_osm_all = slim[slim['source'] == 'osm']
_osm_kept = _osm_all[_osm_all['kept']]
_yes = _osm_tag[_osm_kept.index] == 'yes'
print(f'  ..  OSM fill : {len(_osm_all):,} -> {len(_osm_kept):,} kept; '
      f'{int((_osm_all["drop_reason"] == "list1").sum()):,} dropped by tag (list 1), '
      f'{int((_osm_all["drop_reason"] == "list2_no_poi").sum()):,} dropped by tag with no POI/site (list 2)')
print(f'               kept: {int(_yes.sum()):,} bare yes saved by a POI or site, '
      f'{int((~_yes).sum()):,} with an activity tag or a list-2 tag saved by a POI; '
      f'{int(_osm_kept["has_poi"].sum()):,} of the kept carry a POI')

print()
print('  ..  what the enriched layer now looks like, by activity:')
_ex = (enriched[['building_id', 'volume_3d_m3']]
       .assign(a=enriched['activities'].fillna('').str.split(ALKIS_ACTIVITY_SEP))
       .explode('a'))
_ex['a'] = _ex['a'].str.strip()
_ex = _ex[_ex['a'] != '']
_t2 = _ex.groupby('a').agg(buildings=('building_id', 'size'),
                           volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 1)))
_t2['pct_vol'] = (100 * _t2['volume_Mm3'] /
                  (enriched['volume_3d_m3'].sum() / 1e6)).round(1)
print(_t2.sort_values('buildings', ascending=False).to_string())


LIST 1 - dropped outright (ALKIS_DROP_ALWAYS), biggest first:


  51009_1610   canopy                                  94,571 bld   10.10M m3   153 POIs inside
  OSM          Building mapped in OSM, no ALKIS recor   7,157 bld    0.66M m3    53 POIs inside
  51003_1201   silo                                     1,784 bld    2.44M m3     7 POIs inside
  51002_1250   mast                                     1,772 bld    0.47M m3     0 POIs inside
  51002_1230   Solar cells                              1,177 bld    0.65M m3     3 POIs inside
  51002_1220   Wind turbine                               423 bld    0.87M m3     0 POIs inside
  51003_1205   tank                                       204 bld    0.22M m3     0 POIs inside
  51002_1260   Radio mast                                 178 bld    0.05M m3     1 POIs inside
  51002_1290   chimney                                    119 bld    0.20M m3     2 POIs inside
  51001_1008   Transmission and radio tower                88 bld    0.09M m3     1 POIs inside
  31001_2513   Water container          

  51006_1470   ski jump (inrun)                             4 bld    0.00M m3     3 POIs inside
  51001_1007   Fire lookout tower                           1 bld    0.00M m3     1 POIs inside
  30 classes, 107,923 buildings, 19.2M m3
  ..  of which OSM footprints, by building=* tag (7,157, 53 POIs inside):
        garage                2,301    2 POIs inside
        shed                  1,930    4 POIs inside
        hut                     903   12 POIs inside
        roof                    699   17 POIs inside
        carport                 370    0 POIs inside
        garages                 272    1 POIs inside
        service                 206    7 POIs inside
        allotment_house         198    0 POIs inside
        silo                     32    0 POIs inside
        static_caravan           30    1 POIs inside
        digester                 28    0 POIs inside
        storage_tank             27    1 POIs inside
        ... 22 more tags, 161 footprints
  ..  298 POIs 

  31001_1000   residential buildings                  327,264 bld -> 5,692 kept (4,305 inside, 305 snapped, 1,082 site), 321,572 dropped (279.13M m3)
               POIs on the kept ones: restaurant 312, chalet 231, hairdresser 220, social_facility 203, doctors 203, fast_food 181
  OSM          Building mapped in OSM, no ALKIS recor  42,267 bld -> 1,022 kept (373 inside, 87 snapped, 562 site), 41,245 dropped (7.68M m3)
               POIs on the kept ones: industrial 46, kindergarten 28, fire_station 24, social_facility 24, fast_food 22, ruins 20
  31001_2720   Agricultural and forestry business bui  18,850 bld ->   227 kept (103 inside, 32 snapped, 92 site), 18,623 dropped (27.66M m3)
               POIs on the kept ones: farm 30, equestrian 20, industrial 8, cafe 7, florist 6, garden_centre 4


  31001_2500   Buildings for supply                     7,806 bld ->   333 kept (69 inside, 46 snapped, 218 site), 7,473 dropped (1.11M m3)
               POIs on the kept ones: industrial 45, water_works 26, works 9, commercial 6, university 3, historic 3
  31001_1210   Agricultural and forestry residential    4,197 bld ->    59 kept (41 inside, 11 snapped, 7 site), 4,138 dropped (6.00M m3)
               POIs on the kept ones: farm 8, manor 5, castle 4, cafe 3, restaurant 3, chalet 2
  31001_2600   Building for disposal                      770 bld ->   134 kept (15 inside, 7 snapped, 112 site), 636 dropped (0.33M m3)
               POIs on the kept ones: wastewater_plant 6, industrial 6, commercial 2, recycling 1, company 1, energy_supplier 1
  31001_2740   greenhouse, greenhouse                     602 bld ->    85 kept (13 inside, 2 snapped, 70 site), 517 dropped (0.39M m3)
               POIs on the kept ones: garden_centre 7, florist 3, cafe 2, gardener 2, commercial 1
  31001_3

...

  no OSM footprint   313,065 bld ->    618 kept (521 by a POI or site, 0 by the tag, 97 by a name), 312,447 dropped (33.10M m3)
  bare yes twin      151,553 bld ->  1,084 kept (674 by a POI or site, 0 by the tag, 410 by a name), 150,469 dropped (37.39M m3)
  other twin          52,214 bld ->    469 kept (396 by a POI or site, 0 by the tag, 73 by a name), 51,745 dropped (25.76M m3)
  garage/shed twin    24,625 bld ->      0 kept (0 by a POI or site, 0 by the tag, 0 by a name), 24,625 dropped (2.92M m3)
  activity twin          742 bld ->    642 kept (53 by a POI or site, 589 by the tag, 0 by a name), 100 dropped (0.03M m3)


  ..  539,386 dropped below the floor (99.21M m3, median 31 m2), by class: Buildings for business or commerce 340,418, residential buildings 141,410, Building mapped in OSM, no ALKIS record 29,941, Agricultural and forestry business building 7,695, Buildings for supply 7,064, Commercial and industrial buildings 4,209, Buildings for trade and services 3,161, Buildings for recreational purposes 850
  ..  kept by an ALKIS name alone: Vereinsheim 48, Sportheim 14, Tennisheim 7, Feuerwehr 7, DRK 3, DLRG Rettungsstation 3, Gemeindehaus 3, Post 3, Schießstand 3, Kläranlage 3
  ..  the POIs that kept small buildings, by use: chalet 134, place_of_worship 105, industrial 86, fast_food 82, restaurant 51, fire_station 51, hairdresser 47, kiosk 42, commercial 38, cafe 37

EVIDENCE BAND - 31001_2000 between 100 and 200 m2 (16,735 buildings):
  dropped     16,327
  osm_tag        167
  osm_name       103
  poi             94
  snap            40
  alkis_name       4
  ..  16,327 dropped (11.4M m3); t

LAND-USE RULE - classes 31001_1110, 31001_1120, 31001_1130, 31001_2000, 31001_2010, 31001_2100, 31001_3000, 31001_3200, mute or no twin, no name (25,262 buildings):
  31001_1110: 60 buildings -> 43 kept, 17 dropped (0.0M m3); twin: yes 31, <none> 10, apartments 7, detached 4, house 4, residential 2
  31001_1120: 4,230 buildings -> 3,239 kept, 991 dropped (1.8M m3); twin: yes 2,083, apartments 927, house 531, <none> 495, residential 101, detached 74
  31001_1130: 474 buildings -> 406 kept, 68 dropped (0.1M m3); twin: yes 251, <none> 125, house 50, residential 21, apartments 14, detached 10
  31001_2000: 10,672 buildings -> 3,565 kept, 7,107 dropped (17.1M m3); twin: yes 6,711, <none> 2,368, farm_auxiliary 319, garages 299, barn 258, house 146
  31001_2010: 3,301 buildings -> 3,038 kept, 263 dropped (0.5M m3); twin: yes 1,740, <none> 1,120, apartments 150, house 148, residential 36, detached 26
  31001_2100: 5,633 buildings -> 5,412 kept, 221 dropped (0.4M m3); twin: yes 3,010, <none> 2,

        alkis   869,316 ->   38,186
        osm      50,011 ->    1,600
  ..  rescued   : 10,938 buildings are here only because something spoke for them - 6,115 a POI inside, 854 a snapped POI, 2,526 a site polygon, 756 the activity tag of the OSM twin, 504 a named twin, 183 an ALKIS name (`rescued`, `rescued_by` carried forward)
  ..  OSM fill : 50,011 -> 1,600 kept; 7,157 dropped by tag (list 1), 41,245 dropped by tag with no POI/site (list 2)
               kept: 939 bare yes saved by a POI or site, 661 with an activity tag or a list-2 tag saved by a POI; 1,470 of the kept carry a POI

  ..  what the enriched layer now looks like, by activity:


                 buildings  volume_Mm3  pct_vol
a                                              
work                 31604       248.0     89.5
business             27357       219.0     79.0
meetup               13456        42.5     15.3
errands              12591        58.2     21.0
home                 11356        36.6     13.2
shopping             10051        47.1     17.0
leisure               5338        20.4      7.4
education             2236        17.2      6.2
lessons                732         2.0      0.7
sports                 536         4.6      1.7
early_education        147         1.0      0.4
other                   26         0.0      0.0


## 10. The POI join — which POI belongs to which building

Section 7 decided *whether* a POI is on a building. This section writes down
*which* POI is on *which* building, with the shares the redistribution needs.
Three kinds of pair:

| POI | pair | share |
|---|---|---|
| `point`, `footprint` | the building it was placed on in section 7 — inside, else snapped within `POI_SNAP_MAX_DISTANCE_M` | `share_in_building`: its split area over the sum of split areas of all POIs on that building |
| `unit` | the building containing it, i.e. its mall's | the same, using the `split_area_m2` step 01 computed within the mall |
| `site` | **every** kept building whose representative point lies inside the site polygon — nested sites included, so a building on a campus inside a research park gets both | `share_of_site`: the building's volume over the site's total volume; even when no building in the site has a volume |

**Split area** follows `docs/nested-poi-area-split.md`, applied per building: a
polygon POI uses its own area, a point takes the median area of the polygon
POIs on the same building, and with no polygon at all every point gets
`POI_SPLIT_AREA_FALLBACK_M2` — the even split. Shares sum to 1.0 per building,
asserted.

**The mall gets no share of its own.** A parent whose use is in
`POI_CONTAINER_USES` and whose units are placed is a container: Schloss-Arkaden
is 128 shops, not 128 shops plus a mall. A supermarket with a bakery counter
inside is *not* a container — it keeps a share next to the bakery, because the
supermarket is the main activity. Every other placed POI keeps its parent's use
and name as context (`parent_use`, `parent_name`), so the classification later
sees "Calzedonia, a unit in the mall Schloss-Arkaden".

**A building keeps both** its own POIs and the sites around it. The two share
columns are separate on purpose: `share_in_building` splits the building among
its own POIs; `share_of_site` splits the site's demand among its buildings. How
a campus label and a canteen POI on the same building combine into one demand
is the redistribution's decision, not this notebook's.

**What stays unassigned, and why, is kept** in its own layer: outdoor uses that
never snap, building-bound POIs with nothing within the radius, sites with no
kept building inside, and the container parents. Nothing is silently lost.


In [12]:
kept_ids = set(enriched['building_id'])
_pinfo = pois.set_index('poi_id')

# --- 10a. placed POIs -> one pair each ----------------------------------------
if 'snap_m' not in poi_hits.columns:
    poi_hits['snap_m'] = np.nan
_pl = poi_hits[['poi_id', 'poi_role', 'poi_use', 'name', 'building_id', 'placed_by', 'snap_m']].copy()
_not_kept = ~_pl['building_id'].isin(kept_ids)
if _not_kept.any():
    raise AssertionError(f'{int(_not_kept.sum()):,} placed POIs sit on a building that was dropped - '
                         'a POI on a building must keep it (section 7 rule 2)')
_pl = _pl.merge(pois[['poi_id', 'poi_parent_id', 'split_area_m2', 'area_m2', 'geom_kind']],
                on='poi_id', how='left')
_unit_parents = set(pois.loc[pois['poi_role'] == 'unit', 'poi_parent_id'].dropna())
_container = _pl['poi_use'].isin(POI_CONTAINER_USES) & _pl['poi_id'].isin(_unit_parents)
print(f'  ..  {len(_pl):,} placed POIs; {int(_container.sum())} are container parents '
      f'({", ".join(sorted(POI_CONTAINER_USES))}) whose units are placed - they get no share of their own')
pairs = _pl[~_container].copy()
pairs['how'] = np.where(pairs['poi_role'] == 'unit', 'unit', pairs['placed_by'])

# split area, per building: step 01's value for units, own area for polygons,
# the median of the polygon POIs on the same building for points, else fallback
_sa = pd.to_numeric(pairs['split_area_m2'], errors='coerce')
_from_step01 = _sa.notna()
_own = ~_from_step01 & (pairs['geom_kind'] == 'area')
_sa = _sa.where(~_own, pairs['area_m2'])
_med = _sa.groupby(pairs['building_id']).median()
_sib = _sa.isna() & pairs['building_id'].map(_med).notna()
_sa = _sa.where(_sa.notna(), pairs['building_id'].map(_med))
_fb = _sa.isna()
_sa = _sa.fillna(POI_SPLIT_AREA_FALLBACK_M2)
pairs['split_area_m2'] = _sa.astype(float)
pairs['share_in_building'] = (pairs['split_area_m2']
                              / pairs.groupby('building_id')['split_area_m2'].transform('sum'))
_sum = pairs.groupby('building_id')['share_in_building'].sum()
if not np.allclose(_sum.to_numpy(), 1.0):
    raise AssertionError('share_in_building does not sum to 1.0 on every building')
pairs['parent_use'] = pairs['poi_parent_id'].map(_pinfo['poi_use'])
pairs['parent_name'] = pairs['poi_parent_id'].map(_pinfo['name'])
pairs['share_of_site'] = np.nan
print(f'  ..  {len(pairs):,} building-POI pairs on {pairs["building_id"].nunique():,} buildings: '
      + ', '.join(f'{k} {v:,}' for k, v in pairs['how'].value_counts().items()))
print(f'      split area from: {int(_from_step01.sum()):,} step 01 (units), {int(_own.sum()):,} own polygon area, '
      f'{int(_sib.sum()):,} sibling median, {int(_fb.sum()):,} fallback {POI_SPLIT_AREA_FALLBACK_M2} (even split)')
_per = pairs.groupby('building_id').size()
print('      buildings by number of own POIs: '
      + ', '.join(f'{k} {v:,}' for k, v in pd.cut(_per, [0, 1, 5, 20, 10**6], labels=['1', '2-5', '6-20', '>20'])
                  .value_counts().sort_index().items()))
_busy = _per.sort_values(ascending=False).head(5)
_bname = enriched.set_index('building_id')
for _b, _n in _busy.items():
    _pp = pairs[pairs['building_id'] == _b]
    print(f'        {_b:<18} {_n:>4} POIs  {str(_bname.loc[_b, "label_en"])[:32]:<32} '
          f'top: {", ".join(f"{u} {s:.0%}" for u, s in _pp.nlargest(3, "share_in_building")[["poi_use", "share_in_building"]].itertuples(index=False))}'
          + (f'  in {_pp["parent_name"].dropna().iloc[0]}' if _pp['parent_name'].notna().any() else ''))

# --- 10b. sites -> every kept building inside ----------------------------------
_rep = enriched[['building_id', 'volume_3d_m3', 'geometry']].copy()
_rep['geometry'] = _rep.geometry.representative_point()
_sites = pois.loc[pois['poi_role'] == 'site', ['poi_id', 'poi_use', 'name', 'poi_parent_id', 'geometry']]
sp = gpd.sjoin(_rep, _sites, how='inner', predicate='within')      # NOT deduplicated: nested sites both count
sp = pd.DataFrame(sp.drop(columns=['geometry', 'index_right']))
_vol = sp['volume_3d_m3'].fillna(0.0)
_tot = _vol.groupby(sp['poi_id']).transform('sum')
_n = sp.groupby('poi_id')['building_id'].transform('size')
sp['share_of_site'] = np.where(_tot > 0, _vol / _tot.where(_tot > 0, 1.0), 1.0 / _n)
sp['poi_role'] = 'site'
sp['how'] = 'site'
sp['parent_use'] = sp['poi_parent_id'].map(_pinfo['poi_use'])
sp['parent_name'] = sp['poi_parent_id'].map(_pinfo['name'])
for _c in ('split_area_m2', 'share_in_building', 'snap_m'):
    sp[_c] = np.nan
_ssum = sp.groupby('poi_id')['share_of_site'].sum()
if not np.allclose(_ssum.to_numpy(), 1.0):
    raise AssertionError('share_of_site does not sum to 1.0 on every site')
_multi = (sp.groupby('building_id').size() > 1).sum()
_novol = int((sp.groupby('poi_id')['volume_3d_m3'].sum().fillna(0) == 0).sum())
print()
print(f'  ..  {len(sp):,} building-site pairs: {sp["poi_id"].nunique():,} sites over '
      f'{sp["building_id"].nunique():,} kept buildings; {int(_multi):,} buildings lie in more than one site; '
      f'{_novol} sites have no building with a volume and split evenly')
_empty = _sites[~_sites['poi_id'].isin(sp['poi_id'])]
print(f'      {len(_empty):,} sites contain no kept building and stay unassigned - by use: '
      + ', '.join(f'{k} {v}' for k, v in _empty['poi_use'].value_counts().head(8).items()))
_bigs = sp.groupby('poi_id').agg(n=('building_id', 'size'), name=('name', 'first'), use=('poi_use', 'first')).sort_values('n', ascending=False).head(5)
print('      the biggest sites: ' + ', '.join(f'{r["name"]} ({r["use"]}) {int(r["n"])} buildings' for _, r in _bigs.iterrows()))

# --- 10c. one table of pairs -------------------------------------------------------
_cols = ['building_id', 'poi_id', 'poi_role', 'how', 'poi_use', 'name', 'poi_parent_id', 'parent_use',
         'parent_name', 'split_area_m2', 'share_in_building', 'share_of_site', 'snap_m']
building_pois = pd.concat([pairs[_cols], sp[_cols]], ignore_index=True)
building_pois = building_pois.merge(pois[['poi_id', 'geometry']], on='poi_id', how='left')
building_pois = gpd.GeoDataFrame(building_pois, geometry='geometry', crs=TARGET_CRS)
building_pois['geometry'] = building_pois.geometry.representative_point()
building_pois['snap_m'] = building_pois['snap_m'].round(1)
_key = building_pois['building_id'] + '|' + building_pois['poi_id']
if _key.duplicated().any():
    raise AssertionError(f'{int(_key.duplicated().sum())} duplicate building-POI pairs')
print()
print(f'  ok  building_pois: {len(building_pois):,} pairs, {building_pois["poi_id"].nunique():,} POIs '
      f'on {building_pois["building_id"].nunique():,} buildings')

# --- 10d. the POIs no building got --------------------------------------------------
_un = pois[~pois['poi_id'].isin(building_pois['poi_id'])].copy()
_un['reason'] = np.select(
    [_un['poi_id'].isin(_pl.loc[_container, 'poi_id']),
     _un['poi_role'] == 'site',
     ~_un['poi_use'].isin(_bound_uses)],
    ['container_parent', 'site_without_kept_building', 'outdoor_use'],
    default=f'nothing_within_{POI_SNAP_MAX_DISTANCE_M}m')
pois_unassigned = gpd.GeoDataFrame(_un[['poi_id', 'poi_role', 'poi_use', 'name', 'reason']],
                                   geometry=_un.geometry.representative_point(), crs=TARGET_CRS)
print(f'  ..  {len(_un):,} of {len(pois):,} POIs ({100 * len(_un) / len(pois):.1f} %) are assigned to no building: '
      + ', '.join(f'{k} {v:,}' for k, v in _un['reason'].value_counts().items()))
print('      the building-bound ones with nothing in reach, by use: '
      + ', '.join(f'{k} {v}' for k, v in _un.loc[_un['reason'].str.startswith('nothing'), 'poi_use'].value_counts().head(8).items()))

# --- 10e. the building-side summary --------------------------------------------------
_g = pairs.sort_values('share_in_building', ascending=False).groupby('building_id')
enriched['n_pois'] = enriched['building_id'].map(_g.size()).fillna(0).astype('int64')
enriched['poi_main_use'] = enriched['building_id'].map(_g['poi_use'].first())
enriched['poi_uses'] = enriched['building_id'].map(
    _g['poi_use'].agg(lambda s: ';'.join(dict.fromkeys(s.dropna().astype(str)))))
enriched['poi_names'] = enriched['building_id'].map(
    _g['name'].agg(lambda s: ';'.join(dict.fromkeys(s.dropna().astype(str)))))      # every name, no cap
_gs = sp.groupby('building_id')
enriched['n_sites'] = enriched['building_id'].map(_gs.size()).fillna(0).astype('int64')
enriched['site_uses'] = enriched['building_id'].map(
    _gs['poi_use'].agg(lambda s: ';'.join(dict.fromkeys(s.dropna().astype(str)))))
enriched['site_names'] = enriched['building_id'].map(
    _gs['name'].agg(lambda s: ';'.join(dict.fromkeys(s.dropna().astype(str)))))     # every name, no cap
_has_own, _has_site = enriched['n_pois'] > 0, enriched['n_sites'] > 0
print()
print(f'  ..  buildings with own POIs: {int(_has_own.sum()):,}; inside a site: {int(_has_site.sum()):,}; '
      f'either: {int((_has_own | _has_site).sum()):,} of {len(enriched):,} '
      f'({100 * (_has_own | _has_site).mean():.1f} %) - the rest carry only their ALKIS class')
print('      top poi_main_use: ' + ', '.join(f'{k} {v:,}' for k, v in enriched['poi_main_use'].value_counts().head(10).items()))


  ..  22,652 placed POIs; 19 are container parents (mall) whose units are placed - they get no share of their own
  ..  22,633 building-POI pairs on 17,457 buildings: inside 19,003, snap 2,004, unit 1,626
      split area from: 1,626 step 01 (units), 10,146 own polygon area, 386 sibling median, 10,475 fallback 1.0 (even split)


      buildings by number of own POIs: 1 14,542, 2-5 2,772, 6-20 137, >20 6
        DENIAL0100005xF6    138 POIs  Buildings for trade and services top: castle 10%, electronics 10%, supermarket 4%  in Schloss Braunschweig
        DENIAL9100005MxH     68 POIs  Building for trade and services  top: pharmacy 1%, bakery 1%, fast_food 1%  in City-Galerie Wolfsburg
        DENIAL01000051gc     35 POIs  Commercial and industrial buildi top: engineer 3%, company 3%, gymnastics 3%  in Rolleiwerke-Gelände
        DENIAL0500001k0e     23 POIs  Buildings for trade and services top: pharmacy 4%, bank 4%, clothes 4%  in BraWo Carree Salzgitter
        DENIAL010000dM1V     21 POIs  Buildings for trade and services top: commercial 100%, pharmacy 0%, toys 0%  in BraWoPark Shopping Center



  ..  18,652 building-site pairs: 2,030 sites over 17,911 kept buildings; 741 buildings lie in more than one site; 0 sites have no building with a volume and split evenly
      63 sites contain no kept building and stay unassigned - by use: industrial 16, wastewater_plant 12, plant 7, commercial 6, camp_site 4, grave_yard 3, christian 2, sports_centre 1
      the biggest sites: nan (retail) 442 buildings, Salzgitter Flachstahl GmbH (industrial) 402 buildings, Volkswagenwerk Wolfsburg (factory) 384 buildings, nan (commercial) 325 buildings, Gewerbegebiet Hansestraße Ost (industrial) 293 buildings

  ok  building_pois: 41,285 pairs, 24,663 POIs on 30,103 buildings
  ..  3,180 of 27,843 POIs (11.4 %) are assigned to no building: outdoor_use 2,752, nothing_within_50m 346, site_without_kept_building 63, container_parent 19
      the building-bound ones with nothing in reach, by use: industrial 161, sports_centre 32, commercial 18, historic 17, multi 16, water_works 11, kindergarten 9, eque


  ..  buildings with own POIs: 17,457; inside a site: 17,911; either: 30,103 of 39,786 (75.7 %) - the rest carry only their ALKIS class
      top poi_main_use: restaurant 867, place_of_worship 863, industrial 546, kindergarten 534, fire_station 498, fast_food 404, school 396, hairdresser 388, commercial 382, supermarket 339


## 11. Write the enriched layer

One GeoPackage, `04_buildings_enriched.gpkg`, three layers:

| layer | rows | what |
|---|---|---|
| `buildings` | one per kept building | everything the layer carried — now including `alkis_landuse` and `osm_landuse` under the centre — plus `n_pois`, `poi_main_use`, `poi_uses`, `poi_names`, `n_sites`, `site_uses`, `site_names` — the name lists are complete, no cap: the LLM step wants everything |
| `building_pois` | one per building–POI pair | `how`, the parent context, `share_in_building`, `share_of_site`, `snap_m`; geometry is the POI's point |
| `pois_unassigned` | one per POI on no building | with `reason` |

Nothing is trimmed here. Which columns the LLM step needs and which it does not
is that notebook's first question, and it can only be answered there.

Two QGIS files beside it: `04_poi_assignments.gpkg`, one line per pair from the
POI to the nearest point of its building — long for site pairs, short for
snapped ones, zero-length for a POI inside — and the kept extract from before,
now carrying the POI summary columns so it can be styled by `poi_main_use`.


In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENTAL_DIR.mkdir(parents=True, exist_ok=True)


def _unlink(path):
    if path.exists():
        try:
            path.unlink()
        except PermissionError as e:
            raise RuntimeError(f'{path.name} is locked - close it in QGIS and rerun this cell. '
                               f'Original error: {e}') from None


_unlink(ENRICHED_BUILDINGS_FILE)
t0 = time.perf_counter()
for _layer, _frame in (('buildings', enriched), ('building_pois', building_pois),
                       ('pois_unassigned', pois_unassigned)):
    print(f'Writing {len(_frame):,} rows x {len(_frame.columns)} columns -> {ENRICHED_BUILDINGS_FILE.name} : {_layer} ...',
          flush=True)
    _frame.to_file(ENRICHED_BUILDINGS_FILE, layer=_layer, driver='GPKG')
print(f'  ok  {ENRICHED_BUILDINGS_FILE.stat().st_size / 1e6:,.1f} MB  [{time.perf_counter() - t0:,.0f}s]')
_layers = [l[0] for l in pyogrio.list_layers(ENRICHED_BUILDINGS_FILE)]
if set(_layers) != {'buildings', 'building_pois', 'pois_unassigned'}:
    raise AssertionError(f'unexpected layers in the enriched file: {_layers}')
_chk = gpd.read_file(ENRICHED_BUILDINGS_FILE, layer='building_pois', rows=3)
print(f'  ok  read back {_layers}; building_pois sample:')
print(_chk[['building_id', 'poi_id', 'how', 'poi_use', 'share_in_building', 'share_of_site']].to_string(index=False))

# --- one line per pair, POI -> nearest point of its building, for QGIS -------------
_bg = enriched.set_index('building_id').geometry
_lines = building_pois.copy()
_lines['geometry'] = shapely.shortest_line(_lines.geometry.values, _bg.reindex(_lines['building_id']).values)
_lines['length_m'] = _lines.geometry.length.round(1)
_unlink(POI_ASSIGNMENTS_FILE)
_lines.to_file(POI_ASSIGNMENTS_FILE, layer='assignments', driver='GPKG')
print(f'  ok  {len(_lines):,} assignment lines -> {POI_ASSIGNMENTS_FILE.name} '
      f'({POI_ASSIGNMENTS_FILE.stat().st_size / 1e6:,.1f} MB); style by `how`')


# --- the filtered layer for QGIS: only what survives, now with its POIs ----------
# Same columns as the full review export minus the drop flags, which are
# constant here; class counts and legend recomputed for what is left.
_nk = enriched['function'].value_counts()
enriched['class_label'] = (enriched['function'] + ' · ' + enriched['label_en'].fillna('?')
                           + ' · ' + enriched['function'].map(_nk).map('{:,}'.format))
_kept_cols = ([c for c in inspect_cols if c in enriched.columns and c != 'geometry']
              + ['n_pois', 'poi_main_use', 'poi_uses', 'poi_names', 'n_sites', 'site_uses', 'site_names', 'geometry'])
if KEPT_INSPECT_FILE.exists():
    try:
        KEPT_INSPECT_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{KEPT_INSPECT_FILE.name} is locked - close it in QGIS and rerun '
            f'this cell. Original error: {e}'
        ) from None
print()
print(f'Writing {len(enriched):,} kept rows x {len(_kept_cols)} columns to '
      f'{KEPT_INSPECT_FILE.name} ...', flush=True)
t0 = time.perf_counter()
enriched[_kept_cols].to_file(KEPT_INSPECT_FILE, layer='buildings', driver='GPKG')
print(f'  ok  {KEPT_INSPECT_FILE.stat().st_size / 1e6:,.1f} MB  '
      f'[{time.perf_counter() - t0:,.0f}s]')
_write_qml(enriched, KEPT_INSPECT_FILE.with_suffix('.qml'))
print(f'  ..  in QGIS: add {KEPT_INSPECT_FILE.name} for the filtered-down layer; '
      f'{LABELLED_INSPECT_FILE.name} keeps the dropped ones for comparison')


Writing 39,786 rows x 44 columns -> 04_buildings_enriched.gpkg : buildings ...


Writing 41,285 rows x 14 columns -> 04_buildings_enriched.gpkg : building_pois ...


Writing 3,180 rows x 6 columns -> 04_buildings_enriched.gpkg : pois_unassigned ...


  ok  42.0 MB  [1s]
  ok  read back ['buildings', 'building_pois', 'pois_unassigned']; building_pois sample:
     building_id        poi_id    how     poi_use  share_in_building  share_of_site
DENIAL0300008oSk node/23654979 inside      hostel                1.0            NaN
DENIAL03000097n2 node/31051780 inside guest_house                1.0            NaN
DENIAL0300008H7n node/33779867 inside  restaurant                0.5            NaN


  ok  41,285 assignment lines -> 04_poi_assignments.gpkg (10.4 MB); style by `how`



Writing 39,786 kept rows x 38 columns to 04_buildings_kept.gpkg ...


  ok  32.1 MB  [1s]
  ok  04_buildings_kept.qml: 59 categories, valid XML, 34 KB
  ..  in QGIS: add 04_buildings_kept.gpkg for the filtered-down layer; 04_buildings_labelled.gpkg keeps the dropped ones for comparison


## 12. Where this leaves us

Step 04 is complete and its result is on disk: **`04_buildings_enriched.gpkg`**
with 49,297 buildings, 45,857 building–POI pairs and 3,173 unassigned POIs, on
the OSM snapshot of 2026-09-10. Against the January snapshot that is 844 more
POIs and 7,404 more footprints in, and 331 more buildings out —
businesses mapped this year that the old file did not know.

| | buildings | volume |
|---|---|---|
| in from step 03, plus the OSM fill with its estimated volume | 919,327 | 690.4M m³ |
| list 1 — 29 ALKIS structure classes and 34 OSM tags, dropped outright | −107,923 | −19.2M m³ |
| size floor — under 100 m² with a garage or shed twin, dropped outright | −23,396 | −2.7M m³ |
| list 2 — 19 ALKIS classes and 27 OSM tags incl. the bare `yes`, no POI or site on them | −395,260 | −324.4M m³ |
| size floor — every class under 100 m², no POI, site, activity tag or name | −317,144 | −33.1M m³ |
| evidence band — class 2000 between 100 and 200 m², nothing speaking for it | −16,317 | −11.4M m³ |
| land use — eight noise-prone classes (2000 above 200 m², the others at any size), nothing describes it, on residential or agricultural land | −9,538 | −20.2M m³ |
| twin-structure — every OSM footprint on it a garage, shed or roof, nothing on it | −452 | −0.4M m³ |
| **out** | **49,297** | **279.0M m³** |

Class 2000 alone went from 369,675 to 5,684: 4,696 at or above 200 m² — the
undescribed ones among them only where the ALKIS parcel is commercial,
industrial, public or infrastructure — 403
in the evidence band that something spoke for — an activity tag or a name on the
OSM twin, a POI — and 585 under 100 m² kept by a POI, a site or an activity
twin: car washes, fuel shops, kindergartens, florists.

| the join | |
|---|---|
| POIs on a building of their own | 22,634 pairs on 17,467 buildings — 19,024 inside, 1,984 snapped, 1,626 mall units |
| buildings inside a site | 23,223 pairs, 2,036 sites over 22,293 buildings; 930 buildings in more than one site. Since 2026-09-11 the sites include industrial, commercial and retail land-use areas, named or not: Salzgitter Flachstahl 764 buildings, Volkswagenwerk 496, Gewerbegebiet Hansestraße Ost 315 |
| buildings with any POI evidence | 34,492 of 49,297, 70.0 % — the rest carry only their ALKIS class |
| POIs on no building | 3,173 of 27,843: 2,752 outdoor uses, 345 building-bound with nothing within 50 m, 19 mall containers, 57 empty sites |

9,688 buildings survive only because something spoke for them: 5,892 a POI
inside, 768 a snapped POI, 2,527 a site polygon — mostly buildings of 200 m² or
more standing in an industrial estate — 398 the activity tag of the OSM twin,
103 a name on the twin.

Three review files are in `data/experimental_extract/`:

* **`04_buildings_labelled.gpkg`** — every building including the dropped ones,
  with `kept`, `drop_reason`, `rescued`, `rescued_by`, `n_poi`, `n_poi_snap`,
  `in_site`, `osm_twin_tag`, `osm_twin_name`, `alkis_landuse` and `osm_landuse`,
  so every decision is visible on the map
* **`04_buildings_kept.gpkg`** — only the survivors, with `n_pois`,
  `poi_main_use`, `poi_uses`, `n_sites`, `site_uses`
* **`04_poi_assignments.gpkg`** — one line per pair from the POI to its
  building; style by `how`

### Still open

* **Tourism accommodation rescues homes.** `chalet`, `apartment` and
  `guest_house` POIs keep residential-coded holiday homes. Whether a holiday
  flat is an activity destination is for the classification, not for this
  filter — but it is where those buildings come from.
* **The land decides the last leftovers.** ALKIS Landnutzung, the statewide
  open-data layer cut 2026-01-01, is clipped to the region on every run. It
  removed 9,538 undescribed buildings of the eight noise-prone classes on
  residential and agricultural parcels — 7,115 class 2000, 1,258 *residential
  with trade and services*, 734 *trade and services*, 479 class 2100, 123
  *public purposes*, 98, 50 and 20 in the small ones — and kept the other
  27,046, on business, public and infrastructure land, as generic workplaces. OSM land use
  is on the layer for information only. The same pattern remains in 20
  classes that name a specific use or are business-first with housing, 127
  buildings in all, left alone on purpose.
* **Four thresholds are judgements.** `ALKIS_SIZE_FLOOR_EVIDENCE_M2 = 200`
  on class 2000: between 100 and 200 m² a building needs evidence; 16,317 had
  none and went, 403 stayed. `POI_SITE_RESCUE_MIN_AREA_M2 = 200` keeps 2,527
  buildings; 100 m² would keep about 5,500, no threshold about 16,800. `POI_BUILDING_BOUND_MIN_INSIDE_SHARE = 0.5` lets kindergartens (73 % inside),
  schools (78 %) and sports centres (58 %) snap. `SIZE_FLOOR_M2 = 100`
  on every class: on class 2000 alone, at 50 m² another 53,700 buildings would
  stay, at 75 m² another 13,200. Section 7 prints the site ladder on every run.
* **The two share columns are not yet one weight.** `share_in_building` splits
  a building among its own POIs, `share_of_site` splits a site among its
  buildings. How a campus label and a canteen POI on the same building combine
  is the redistribution's decision.
* **The 1,609 OSM footprints carry an estimated volume.** The previous
  pipeline's rule: `height` tag, else `building:levels` × 2 m, else one floor of
  2 m; 84 % of them get the default, so their weight is small and marked by
  `source == 'osm'`. They still need an OSM `building=*` → activity map.

### Next

A new notebook that prepares the LLM input. Its first section decides, per
column, what the LLM needs, what only the model needs, and what is noise. The
name and use lists on the buildings are complete on purpose — no cap — because
the LLM works better with more context, not less.
